# 10 — Edición y generación de las cinco figuras cuantitativas finales

Este notebook es **portátil y autónomo**: incorpora las tablas finales que
necesitan las cinco figuras y no busca ninguna carpeta de proyecto. Cada figura
tiene su código de Matplotlib íntegramente visible, con el objeto `fig`, sus
ejes, títulos, colores, tamaños, anotaciones, leyendas y exportación.

Las tablas embebidas corresponden a los resultados definitivos: firma
Basal-like direccional corregida, score biológico con coeficiente 0.075,
concordancia externa, comparadores basados en q, sensibilidad a kappa y 1,000
rewirings. El notebook no reentrena el modelo ni recalcula pruebas estadísticas.

In [ ]:
from pathlib import Path
import json
import math
import os

os.environ.setdefault('MPLCONFIGDIR', '/tmp/intergate-mpl')

import matplotlib.pyplot as plt
from matplotlib import patheffects
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, Rectangle
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# ===================== EDITAR AQUÍ: CARPETA DE SALIDA =====================
# Por defecto crea ./figures respecto al directorio desde el que se abrió
# Jupyter. Puede sustituirse por cualquier ruta local mediante Path(...).
OUTPUT_ROOT = Path.cwd()
FIGURES = OUTPUT_ROOT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

print('Salida de figuras:', FIGURES.resolve())

## Estilo global y carga de datos

Modifica esta celda para cambiar conjuntamente tipografía, tamaños, paleta,
resolución y dimensiones. Los parámetros específicos permanecen al principio
de la celda de cada figura.

In [ ]:
# ===================== EDITAR AQUÍ: ESTILO GLOBAL =====================
STYLE = {
    'font_family': 'DejaVu Sans',
    'base_font_size': 10.0,
    'figure_title_size': 16,
    'panel_title_size': 11.5,
    'axis_label_size': 10.5,
    'tick_label_size': 9.5,
    'legend_size': 8.5,
    'annotation_size': 8.5,
    'gene_label_size': 8.0,
    'dpi': 600,
}

CLASS_COLORS = {
    'LumA': '#4C78A8',
    'LumB': '#72B7B2',
    'HER2-enriched': '#F58518',
    'Basal-like': '#C44E52',
}
NEUTRAL = '#BDBDBD'
DARK_NEUTRAL = '#4D4D4D'
HEATMAP_CMAP = 'magma'
DIAGONAL_COLOR = '#00BFC4'

FIGSIZE = {
    'structural': (12.2, 8.6),
    'basal_core': (14.2, 8.2),
    'biological_validation': (12.4, 8.8),
    'external_concordance': (8.4, 5.0),
    'controls_sensitivity': (15.0, 8.8),
}

plt.rcParams.update({
    'font.family': STYLE['font_family'],
    'font.size': STYLE['base_font_size'],
    'axes.titleweight': 'bold',
    'axes.labelsize': STYLE['axis_label_size'],
    'axes.titlesize': STYLE['panel_title_size'],
    'xtick.labelsize': STYLE['tick_label_size'],
    'ytick.labelsize': STYLE['tick_label_size'],
    'legend.fontsize': STYLE['legend_size'],
    'axes.grid': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

def export_figure(fig, stem):
    """Exporta de forma atómica y mantiene la figura abierta para editarla."""
    for extension in ('pdf', 'svg', 'png'):
        target = FIGURES / f'{stem}.{extension}'
        temporary = FIGURES / f'.{stem}.{extension}.tmp'
        options = {'format': extension, 'bbox_inches': 'tight'}
        if extension == 'png':
            options['dpi'] = STYLE['dpi']
        fig.savefig(temporary, **options)
        os.replace(temporary, target)

def component_sets(edge_pairs):
    """Componentes débiles de una lista de pares dirigidos."""
    nodes = sorted({node for edge in edge_pairs for node in edge})
    parent = {node: node for node in nodes}

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left, right):
        root_left, root_right = find(left), find(right)
        if root_left != root_right:
            parent[root_right] = root_left

    for left, right in edge_pairs:
        union(left, right)
    groups = {}
    for node in nodes:
        groups.setdefault(find(node), set()).add(node)
    return sorted(groups.values(), key=lambda values: (-len(values), sorted(values)))

def force_layout(nodes, edges, seed=42):
    """Layout Fruchterman–Reingold pequeño y determinista."""
    rng = np.random.default_rng(seed)
    n_nodes = len(nodes)
    index = {node: position for position, node in enumerate(nodes)}
    positions = rng.normal(0, 0.35, size=(n_nodes, 2))
    optimal_distance = 1.25 / math.sqrt(max(n_nodes, 1))
    temperature = 0.14
    for step in range(320):
        delta = positions[:, None, :] - positions[None, :, :]
        distance = np.sqrt((delta**2).sum(axis=2)) + np.eye(n_nodes)
        repulsion = (
            (delta / distance[:, :, None])
            * ((optimal_distance**2) / np.maximum(distance, 1e-3))[:, :, None]
        )
        force = repulsion.sum(axis=1)
        for source, target, _ in edges:
            source_index, target_index = index[source], index[target]
            vector = positions[source_index] - positions[target_index]
            length = max(float(np.linalg.norm(vector)), 1e-3)
            attraction = (vector / length) * (length**2 / optimal_distance)
            force[source_index] -= attraction
            force[target_index] += attraction
        norms = np.linalg.norm(force, axis=1)
        positions += (
            force / np.maximum(norms[:, None], 1e-9)
            * np.minimum(norms, temperature)[:, None]
        )
        positions -= positions.mean(axis=0)
        temperature = 0.14 * (1 - (step + 1) / 330)
    span = np.ptp(positions, axis=0)
    positions = positions / np.maximum(span, 1e-9)
    return {node: positions[index[node]] for node in nodes}

def clean_term(term):
    return (
        term.replace('HALLMARK_', '')
        .replace('_', ' ')
        .replace('PI3K AKT MTOR', 'PI3K-AKT-mTOR')
        .replace('G2M', 'G2/M')
        .title()
        .replace('Pi3K-Akt-Mtor', 'PI3K-AKT-mTOR')
        .replace('Myc', 'MYC')
        .replace('E2F', 'E2F')
        .replace('G2/M', 'G2/M')
    )

CLASSES = ['LumA', 'LumB', 'HER2-enriched', 'Basal-like']
DISPLAY_CLASSES = ['LumA', 'LumB', 'HER2', 'Basal-like']
EDGE_TYPES = {0: 'HuRI', 1: 'OmniPath activation', 2: 'OmniPath inhibition'}

# Datos finales embebidos. No es necesario editar este bloque.
FIGURE_DATA = json.loads(r"""{"summary_structural":[{"class_name":"LumA","exact_k_edges":529,"union_edges_across_seeds":928,"mean_pairwise_edge_jaccard":0.5127853883,"strict_5of5_directed_edges":227,"primary_edge_supported_genes":221,"secondary_node_presence_genes":232,"consensus_nodes":517,"consensus_weak_components":96,"consensus_giant_nodes":189,"consensus_giant_fraction":0.3655705996,"consensus_isolated_one_edge_components":43,"consensus_degree_one_nodes":282,"strict_core_nodes":221,"strict_core_weak_components":47,"strict_core_giant_nodes":68,"strict_core_giant_fraction":0.3076923077,"strict_core_isolated_one_edge_components":23,"strict_core_degree_one_nodes":121},{"class_name":"LumB","exact_k_edges":529,"union_edges_across_seeds":845,"mean_pairwise_edge_jaccard":0.5704036192,"strict_5of5_directed_edges":245,"primary_edge_supported_genes":218,"secondary_node_presence_genes":230,"consensus_nodes":504,"consensus_weak_components":102,"consensus_giant_nodes":102,"consensus_giant_fraction":0.2023809524,"consensus_isolated_one_edge_components":44,"consensus_degree_one_nodes":250,"strict_core_nodes":218,"strict_core_weak_components":48,"strict_core_giant_nodes":29,"strict_core_giant_fraction":0.1330275229,"strict_core_isolated_one_edge_components":11,"strict_core_degree_one_nodes":82},{"class_name":"HER2-enriched","exact_k_edges":529,"union_edges_across_seeds":865,"mean_pairwise_edge_jaccard":0.588507739,"strict_5of5_directed_edges":288,"primary_edge_supported_genes":300,"secondary_node_presence_genes":312,"consensus_nodes":536,"consensus_weak_components":124,"consensus_giant_nodes":38,"consensus_giant_fraction":0.0708955224,"consensus_isolated_one_edge_components":53,"consensus_degree_one_nodes":277,"strict_core_nodes":300,"strict_core_weak_components":87,"strict_core_giant_nodes":20,"strict_core_giant_fraction":0.0666666667,"strict_core_isolated_one_edge_components":33,"strict_core_degree_one_nodes":139},{"class_name":"Basal-like","exact_k_edges":529,"union_edges_across_seeds":825,"mean_pairwise_edge_jaccard":0.6039258141,"strict_5of5_directed_edges":284,"primary_edge_supported_genes":257,"secondary_node_presence_genes":267,"consensus_nodes":480,"consensus_weak_components":87,"consensus_giant_nodes":209,"consensus_giant_fraction":0.4354166667,"consensus_isolated_one_edge_components":42,"consensus_degree_one_nodes":239,"strict_core_nodes":257,"strict_core_weak_components":52,"strict_core_giant_nodes":84,"strict_core_giant_fraction":0.326848249,"strict_core_isolated_one_edge_components":19,"strict_core_degree_one_nodes":118}],"components":[{"component_rank":1,"n_genes":84,"n_directed_edges":130,"basal_signature_genes_n":16,"basal_signature_genes":"CDC5L;CDK1;CYB5R2;E2F3;FOXM1;KRT16;KRT4;LMO4;PFKP;PML;SOX10;TEAD4;TFDP1;TRIP13;VGLL1;YES1","significant_hallmark_overlap":"E2F Targets (8); G2/M Checkpoint (10); MYC Targets V1 (4); Notch Signaling (3); PI3K-AKT-mTOR Signaling (6); Wnt Beta Catenin Signaling (1)","genes":"ACTN4;AES;AGR2;ANAPC1;ATG5;AURKB;C1ORF109;C21ORF91;CCDC53;CCHCR1;CCNB1;CCNC;CCND1;CDC5L;CDCA4;CDK1;CDK19;CEP76;CHEK2;CHERP;CSNK2B;CYB5R2;DAZAP2;E2F1;E2F3;EGFR;FAM64A;FAM90A1;FANCI;FOXM1;GOLGA2;GRB2;HES1;HGS;HHEX;INCENP;JRK;KCTD9;KDM1A;KRT16;KRT18;KRT4;KRT5;LASP1;LMNB1;LMNB2;LMO4;MICALL2;NAP1L2;NCAPH2;NOTCH1;NUP54;NUTF2;OPTN;PCNA;PFDN2;PFDN5;PFKP;PLEKHG6;PLK1;PLOD3;PML;POLH;PSMB4;PTK2;RB1;RINT1;SCNM1;SNRPA;SOX10;STRAP;SUOX;TCEA2;TEAD4;TENC1;TFDP1;TP53BP2;TRAF2;TRIM27;TRIP13;USP1;VGLL1;YES1;ZNF688"},{"component_rank":2,"n_genes":9,"n_directed_edges":8,"basal_signature_genes_n":3,"basal_signature_genes":"CDKN2A;ODC1;SKP2","significant_hallmark_overlap":"E2F Targets (4); G2/M Checkpoint (4); MYC Targets V1 (3); Notch Signaling (1); PI3K-AKT-mTOR Signaling (1); Wnt Beta Catenin Signaling (3)","genes":"AKT1;CDKN2A;CUL1;EZH2;INPP4B;MYC;ODC1;SKP2;TFRC"},{"component_rank":3,"n_genes":9,"n_directed_edges":8,"basal_signature_genes_n":1,"basal_signature_genes":"ATAD2","significant_hallmark_overlap":"E2F Targets (1); MYC Targets V1 (1); PI3K-AKT-mTOR Signaling (2)","genes":"AR;ATAD2;CASP7;CDK2;ESR1;MAPK1;PCBP2;TUB;ZNF318"},{"component_rank":4,"n_genes":9,"n_directed_edges":10,"basal_signature_genes_n":6,"basal_signature_genes":"CHAF1A;GRHL2;PAX6;SET;SIX3;ZNF124","significant_hallmark_overlap":"E2F Targets (2); G2/M Checkpoint (3); MYC Targets V1 (2)","genes":"CHAF1A;CKS1B;GRHL2;HMGB3;HNRNPC;PAX6;SET;SIX3;ZNF124"},{"component_rank":5,"n_genes":8,"n_directed_edges":12,"basal_signature_genes_n":2,"basal_signature_genes":"BYSL;OSBPL3","significant_hallmark_overlap":"","genes":"BHLHE40;BYSL;CDK18;MPHOSPH10;OSBPL3;PRPF3;RABEP1;VIM"},{"component_rank":6,"n_genes":7,"n_directed_edges":11,"basal_signature_genes_n":1,"basal_signature_genes":"KCNK5","significant_hallmark_overlap":"Notch Signaling (1)","genes":"ABHD4;APH1A;FXYD6;IGFBP5;KCNK5;LHFPL2;REEP5"},{"component_rank":7,"n_genes":6,"n_directed_edges":5,"basal_signature_genes_n":1,"basal_signature_genes":"WNT5B","significant_hallmark_overlap":"Wnt Beta Catenin Signaling (2)","genes":"AXIN1;CSNK1A1;CSNK1G3;DVL1;LRP6;WNT5B"},{"component_rank":8,"n_genes":6,"n_directed_edges":6,"basal_signature_genes_n":2,"basal_signature_genes":"MDC1;MSH6","significant_hallmark_overlap":"E2F Targets (1)","genes":"BLM;CLK2;MDC1;MSH2;MSH6;RNF8"},{"component_rank":9,"n_genes":6,"n_directed_edges":5,"basal_signature_genes_n":1,"basal_signature_genes":"KLHL24","significant_hallmark_overlap":"","genes":"FAM53C;KLHL24;NCK2;NET1;SCML2;SF3B4"},{"component_rank":10,"n_genes":6,"n_directed_edges":7,"basal_signature_genes_n":2,"basal_signature_genes":"TBX19;ZIC1","significant_hallmark_overlap":"","genes":"HRASLS;POGZ;TBX19;TFG;TRIP6;ZIC1"},{"component_rank":11,"n_genes":5,"n_directed_edges":7,"basal_signature_genes_n":1,"basal_signature_genes":"PDE9A","significant_hallmark_overlap":"","genes":"CRY2;IQCB1;PDE9A;SPG21;TRIM32"},{"component_rank":12,"n_genes":5,"n_directed_edges":8,"basal_signature_genes_n":1,"basal_signature_genes":"TIMM44","significant_hallmark_overlap":"","genes":"SAR1B;SFT2D2;SLC16A10;TIMM44;TMEM14B"},{"component_rank":13,"n_genes":4,"n_directed_edges":3,"basal_signature_genes_n":1,"basal_signature_genes":"CCKBR","significant_hallmark_overlap":"","genes":"CCKBR;GNAI3;GNAZ;OPRK1"},{"component_rank":14,"n_genes":4,"n_directed_edges":5,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"CDC25C;CEP72;CHCHD3;PPP1CB"},{"component_rank":15,"n_genes":4,"n_directed_edges":3,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"PI3K-AKT-mTOR Signaling (1)","genes":"MKNK1;PPP2CA;PRKD1;VASP"},{"component_rank":16,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":1,"basal_signature_genes":"TRIM29","significant_hallmark_overlap":"","genes":"ANKS1A;GPRIN2;TRIM29"},{"component_rank":17,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"E2F Targets (1); G2/M Checkpoint (1)","genes":"ARHGEF2;AURKA;PARD3"},{"component_rank":18,"n_genes":3,"n_directed_edges":3,"basal_signature_genes_n":1,"basal_signature_genes":"RIPK4","significant_hallmark_overlap":"","genes":"BIRC2;RIPK4;RUNX1T1"},{"component_rank":19,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":1,"basal_signature_genes":"GPR161","significant_hallmark_overlap":"G2/M Checkpoint (1)","genes":"EFNA5;GNAS;GPR161"},{"component_rank":20,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":2,"basal_signature_genes":"FZD7;SLC6A15","significant_hallmark_overlap":"Notch Signaling (1)","genes":"FZD7;HPCAL1;SLC6A15"},{"component_rank":21,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":2,"basal_signature_genes":"GSTP1;KRT83","significant_hallmark_overlap":"","genes":"GSTP1;KRT83;TGM1"},{"component_rank":22,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":1,"basal_signature_genes":"KRT6B","significant_hallmark_overlap":"","genes":"KRT13;KRT6B;NUP62"},{"component_rank":23,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":1,"basal_signature_genes":"PRKD3","significant_hallmark_overlap":"PI3K-AKT-mTOR Signaling (1)","genes":"MFF;PAK4;PRKD3"},{"component_rank":24,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":1,"basal_signature_genes":"MID1","significant_hallmark_overlap":"MYC Targets V1 (1)","genes":"MID1;UBE2D2;UBE2L3"},{"component_rank":25,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"RNF139;SREBF1;SREBF2"},{"component_rank":26,"n_genes":3,"n_directed_edges":2,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"PI3K-AKT-mTOR Signaling (1)","genes":"SIKE1;TBK1;TNIP1"},{"component_rank":27,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":1,"basal_signature_genes":"AKT3","significant_hallmark_overlap":"","genes":"AKT3;PHLPP2"},{"component_rank":28,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"BCL2;BCL2L11"},{"component_rank":29,"n_genes":2,"n_directed_edges":2,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"G2/M Checkpoint (1)","genes":"CASP8AP2;NAA10"},{"component_rank":30,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":2,"basal_signature_genes":"CDC25A;CDK6","significant_hallmark_overlap":"E2F Targets (1); G2/M Checkpoint (1)","genes":"CDC25A;CDK6"},{"component_rank":31,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":1,"basal_signature_genes":"CRLF1","significant_hallmark_overlap":"","genes":"CNTFR;CRLF1"},{"component_rank":32,"n_genes":2,"n_directed_edges":2,"basal_signature_genes_n":1,"basal_signature_genes":"ITGB8","significant_hallmark_overlap":"","genes":"CSK;ITGB8"},{"component_rank":33,"n_genes":2,"n_directed_edges":2,"basal_signature_genes_n":1,"basal_signature_genes":"DMD","significant_hallmark_overlap":"G2/M Checkpoint (1)","genes":"DMD;DTNB"},{"component_rank":34,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"DNM1L;MTOR"},{"component_rank":35,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"EFNA3;EPHB1"},{"component_rank":36,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":1,"basal_signature_genes":"ELF5","significant_hallmark_overlap":"","genes":"ELF5;GLRX2"},{"component_rank":37,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"EXOSC4;PPARA"},{"component_rank":38,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"HNRNPH1;RBM38"},{"component_rank":39,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"E2F Targets (1)","genes":"HNRNPK;TK1"},{"component_rank":40,"n_genes":2,"n_directed_edges":2,"basal_signature_genes_n":1,"basal_signature_genes":"PELI1","significant_hallmark_overlap":"","genes":"IRAK1;PELI1"},{"component_rank":41,"n_genes":2,"n_directed_edges":2,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"E2F Targets (1); G2/M Checkpoint (1); MYC Targets V1 (1)","genes":"KPNA2;RNMT"},{"component_rank":42,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"KRT14;KRT86"},{"component_rank":43,"n_genes":2,"n_directed_edges":2,"basal_signature_genes_n":2,"basal_signature_genes":"LCN2;VEGFA","significant_hallmark_overlap":"","genes":"LCN2;VEGFA"},{"component_rank":44,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"NOTCH2NL;STAC"},{"component_rank":45,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"G2/M Checkpoint (1); MYC Targets V1 (1)","genes":"PAX3;RAD23B"},{"component_rank":46,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"PCID2;SHFM1"},{"component_rank":47,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"PLXND1;SEMA3E"},{"component_rank":48,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":1,"basal_signature_genes":"RAB6B","significant_hallmark_overlap":"","genes":"PMM1;RAB6B"},{"component_rank":49,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"PTPRG;SHC1"},{"component_rank":50,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":1,"basal_signature_genes":"PVR","significant_hallmark_overlap":"","genes":"PVR;TSPO"},{"component_rank":51,"n_genes":2,"n_directed_edges":2,"basal_signature_genes_n":2,"basal_signature_genes":"S100A1;S100B","significant_hallmark_overlap":"","genes":"S100A1;S100B"},{"component_rank":52,"n_genes":2,"n_directed_edges":1,"basal_signature_genes_n":0,"basal_signature_genes":"","significant_hallmark_overlap":"","genes":"TERF1;TNKS"}],"basal_strict_edges":[{"source_gene":"CEP76","target_gene":"PLK1","edge_type":2,"median_differential_logit":0.031549722},{"source_gene":"DMD","target_gene":"DTNB","edge_type":0,"median_differential_logit":0.0315625072},{"source_gene":"PPP1CB","target_gene":"CDC25C","edge_type":1,"median_differential_logit":0.0315306336},{"source_gene":"ZIC1","target_gene":"TBX19","edge_type":0,"median_differential_logit":0.0315260291},{"source_gene":"EGFR","target_gene":"PCNA","edge_type":1,"median_differential_logit":0.0315279961},{"source_gene":"SFT2D2","target_gene":"TMEM14B","edge_type":0,"median_differential_logit":0.0315275043},{"source_gene":"VGLL1","target_gene":"TEAD4","edge_type":0,"median_differential_logit":0.0315280408},{"source_gene":"FAM64A","target_gene":"CSNK2B","edge_type":0,"median_differential_logit":0.0315247774},{"source_gene":"TFDP1","target_gene":"E2F1","edge_type":1,"median_differential_logit":0.0315269828},{"source_gene":"NCK2","target_gene":"SCML2","edge_type":0,"median_differential_logit":0.0315303206},{"source_gene":"S100B","target_gene":"S100A1","edge_type":0,"median_differential_logit":0.0315234661},{"source_gene":"SFT2D2","target_gene":"TIMM44","edge_type":0,"median_differential_logit":0.0315268338},{"source_gene":"PLK1","target_gene":"ANAPC1","edge_type":1,"median_differential_logit":0.0315270871},{"source_gene":"PDE9A","target_gene":"IQCB1","edge_type":0,"median_differential_logit":0.0315255821},{"source_gene":"IQCB1","target_gene":"PDE9A","edge_type":0,"median_differential_logit":0.0315261632},{"source_gene":"MYC","target_gene":"ODC1","edge_type":1,"median_differential_logit":0.0315260291},{"source_gene":"PLOD3","target_gene":"KCTD9","edge_type":0,"median_differential_logit":0.0315292031},{"source_gene":"PAX6","target_gene":"CHAF1A","edge_type":0,"median_differential_logit":0.0315255225},{"source_gene":"ANKS1A","target_gene":"GPRIN2","edge_type":0,"median_differential_logit":0.0315252692},{"source_gene":"LCN2","target_gene":"VEGFA","edge_type":0,"median_differential_logit":0.0315260887},{"source_gene":"CASP7","target_gene":"AR","edge_type":2,"median_differential_logit":0.0315327048},{"source_gene":"MYC","target_gene":"CUL1","edge_type":1,"median_differential_logit":0.0315262973},{"source_gene":"VGLL1","target_gene":"C1ORF109","edge_type":0,"median_differential_logit":0.0315276682},{"source_gene":"OPRK1","target_gene":"GNAI3","edge_type":1,"median_differential_logit":0.0315268934},{"source_gene":"PELI1","target_gene":"IRAK1","edge_type":1,"median_differential_logit":0.031526804},{"source_gene":"IRAK1","target_gene":"PELI1","edge_type":1,"median_differential_logit":0.0315252393},{"source_gene":"SKP2","target_gene":"MYC","edge_type":1,"median_differential_logit":0.0315264463},{"source_gene":"TNKS","target_gene":"TERF1","edge_type":2,"median_differential_logit":0.0315271318},{"source_gene":"PPP1CB","target_gene":"CHCHD3","edge_type":0,"median_differential_logit":0.0315273106},{"source_gene":"KCTD9","target_gene":"PLOD3","edge_type":0,"median_differential_logit":0.0315254927},{"source_gene":"GSTP1","target_gene":"TGM1","edge_type":0,"median_differential_logit":0.0315309763},{"source_gene":"CASP8AP2","target_gene":"NAA10","edge_type":0,"median_differential_logit":0.0315251946},{"source_gene":"TBX19","target_gene":"TFG","edge_type":0,"median_differential_logit":0.0315250158},{"source_gene":"AURKA","target_gene":"ARHGEF2","edge_type":2,"median_differential_logit":0.0315276384},{"source_gene":"IQCB1","target_gene":"SPG21","edge_type":0,"median_differential_logit":0.0315271467},{"source_gene":"CRLF1","target_gene":"CNTFR","edge_type":1,"median_differential_logit":0.0315256119},{"source_gene":"KRT16","target_gene":"KRT4","edge_type":0,"median_differential_logit":0.0315251797},{"source_gene":"EFNA3","target_gene":"EPHB1","edge_type":1,"median_differential_logit":0.0315256417},{"source_gene":"GPR161","target_gene":"GNAS","edge_type":1,"median_differential_logit":0.0315262377},{"source_gene":"RAB6B","target_gene":"PMM1","edge_type":0,"median_differential_logit":0.031527102},{"source_gene":"NCK2","target_gene":"KLHL24","edge_type":0,"median_differential_logit":0.0315272659},{"source_gene":"PCID2","target_gene":"SHFM1","edge_type":0,"median_differential_logit":0.0315268636},{"source_gene":"KPNA2","target_gene":"RNMT","edge_type":2,"median_differential_logit":0.0315251946},{"source_gene":"WNT5B","target_gene":"LRP6","edge_type":1,"median_differential_logit":0.0315292478},{"source_gene":"CDK19","target_gene":"NOTCH1","edge_type":2,"median_differential_logit":0.0315218568},{"source_gene":"CDK1","target_gene":"PML","edge_type":2,"median_differential_logit":0.0315236151},{"source_gene":"TRIP13","target_gene":"EGFR","edge_type":1,"median_differential_logit":0.0315249264},{"source_gene":"KRT6B","target_gene":"NUP62","edge_type":0,"median_differential_logit":0.0315321535},{"source_gene":"EGFR","target_gene":"TRIP13","edge_type":1,"median_differential_logit":0.0315236151},{"source_gene":"BYSL","target_gene":"MPHOSPH10","edge_type":0,"median_differential_logit":0.0315254778},{"source_gene":"RAD23B","target_gene":"PAX3","edge_type":2,"median_differential_logit":0.0315257311},{"source_gene":"HNRNPK","target_gene":"TK1","edge_type":2,"median_differential_logit":0.0315257013},{"source_gene":"CDK1","target_gene":"EGFR","edge_type":2,"median_differential_logit":0.0315218568},{"source_gene":"FZD7","target_gene":"HPCAL1","edge_type":0,"median_differential_logit":0.0315493643},{"source_gene":"PPP2CA","target_gene":"PRKD1","edge_type":2,"median_differential_logit":0.0315325409},{"source_gene":"RNF8","target_gene":"BLM","edge_type":1,"median_differential_logit":0.0315236896},{"source_gene":"PTPRG","target_gene":"SHC1","edge_type":2,"median_differential_logit":0.0315279067},{"source_gene":"INPP4B","target_gene":"AKT1","edge_type":2,"median_differential_logit":0.0315303206},{"source_gene":"KRT6B","target_gene":"KRT13","edge_type":0,"median_differential_logit":0.0315320939},{"source_gene":"PHLPP2","target_gene":"AKT3","edge_type":2,"median_differential_logit":0.031528011},{"source_gene":"YES1","target_gene":"TRAF2","edge_type":0,"median_differential_logit":0.0315260291},{"source_gene":"ITGB8","target_gene":"CSK","edge_type":1,"median_differential_logit":0.0315344483},{"source_gene":"TBK1","target_gene":"TNIP1","edge_type":2,"median_differential_logit":0.031532526},{"source_gene":"ZIC1","target_gene":"TRIP6","edge_type":0,"median_differential_logit":0.0315247476},{"source_gene":"PAX6","target_gene":"HMGB3","edge_type":0,"median_differential_logit":0.0315243006},{"source_gene":"CSK","target_gene":"ITGB8","edge_type":1,"median_differential_logit":0.0315266699},{"source_gene":"TP53BP2","target_gene":"LMNB1","edge_type":0,"median_differential_logit":0.0315252244},{"source_gene":"KRT14","target_gene":"KRT86","edge_type":0,"median_differential_logit":0.0315317214},{"source_gene":"KRT5","target_gene":"KRT16","edge_type":0,"median_differential_logit":0.0315325707},{"source_gene":"RNF139","target_gene":"SREBF1","edge_type":2,"median_differential_logit":0.0315244347},{"source_gene":"PPARA","target_gene":"EXOSC4","edge_type":0,"median_differential_logit":0.0315299481},{"source_gene":"MYC","target_gene":"CDKN2A","edge_type":1,"median_differential_logit":0.0315249562},{"source_gene":"KRT16","target_gene":"HGS","edge_type":0,"median_differential_logit":0.0315243602},{"source_gene":"PTK2","target_gene":"ACTN4","edge_type":1,"median_differential_logit":0.0315244645},{"source_gene":"OSBPL3","target_gene":"BYSL","edge_type":0,"median_differential_logit":0.0315233469},{"source_gene":"MTOR","target_gene":"DNM1L","edge_type":1,"median_differential_logit":0.0315248668},{"source_gene":"TSPO","target_gene":"PVR","edge_type":0,"median_differential_logit":0.0315313339},{"source_gene":"PAX6","target_gene":"GRHL2","edge_type":0,"median_differential_logit":0.0315249264},{"source_gene":"SEMA3E","target_gene":"PLXND1","edge_type":1,"median_differential_logit":0.0315256417},{"source_gene":"ELF5","target_gene":"GLRX2","edge_type":0,"median_differential_logit":0.0315277725},{"source_gene":"RNF139","target_gene":"SREBF2","edge_type":2,"median_differential_logit":0.0315256119},{"source_gene":"GSTP1","target_gene":"KRT83","edge_type":0,"median_differential_logit":0.0315300524},{"source_gene":"CSNK2B","target_gene":"CHEK2","edge_type":2,"median_differential_logit":0.0315281153},{"source_gene":"TRIM29","target_gene":"ANKS1A","edge_type":0,"median_differential_logit":0.0315249562},{"source_gene":"JRK","target_gene":"KDM1A","edge_type":0,"median_differential_logit":0.0315292478},{"source_gene":"CCKBR","target_gene":"GNAI3","edge_type":1,"median_differential_logit":0.0315237492},{"source_gene":"BCL2L11","target_gene":"BCL2","edge_type":2,"median_differential_logit":0.0315253437},{"source_gene":"PCNA","target_gene":"POLH","edge_type":1,"median_differential_logit":0.0315254182},{"source_gene":"RBM38","target_gene":"HNRNPH1","edge_type":0,"median_differential_logit":0.0315262526},{"source_gene":"FZD7","target_gene":"SLC6A15","edge_type":0,"median_differential_logit":0.0315494388},{"source_gene":"KRT16","target_gene":"CCNC","edge_type":0,"median_differential_logit":0.0315242112},{"source_gene":"FXYD6","target_gene":"APH1A","edge_type":0,"median_differential_logit":0.0315267444},{"source_gene":"PRKD3","target_gene":"MFF","edge_type":1,"median_differential_logit":0.0315307081},{"source_gene":"NCK2","target_gene":"NET1","edge_type":0,"median_differential_logit":0.0315289348},{"source_gene":"EFNA5","target_gene":"GPR161","edge_type":0,"median_differential_logit":0.0315317214},{"source_gene":"PAX6","target_gene":"HNRNPC","edge_type":0,"median_differential_logit":0.0315240026},{"source_gene":"MID1","target_gene":"UBE2D2","edge_type":0,"median_differential_logit":0.0315289944},{"source_gene":"TUB","target_gene":"MAPK1","edge_type":1,"median_differential_logit":0.0315267742},{"source_gene":"BIRC2","target_gene":"RIPK4","edge_type":1,"median_differential_logit":0.0315291882},{"source_gene":"OSBPL3","target_gene":"PRPF3","edge_type":0,"median_differential_logit":0.031523481},{"source_gene":"PLK1","target_gene":"NCAPH2","edge_type":1,"median_differential_logit":0.0315277278},{"source_gene":"PFKP","target_gene":"NUTF2","edge_type":0,"median_differential_logit":0.0315249115},{"source_gene":"STAC","target_gene":"NOTCH2NL","edge_type":0,"median_differential_logit":0.0315259546},{"source_gene":"OPRK1","target_gene":"GNAZ","edge_type":1,"median_differential_logit":0.0315243453},{"source_gene":"LRP6","target_gene":"DVL1","edge_type":1,"median_differential_logit":0.0315258801},{"source_gene":"NCK2","target_gene":"FAM53C","edge_type":0,"median_differential_logit":0.0315279663},{"source_gene":"TBX19","target_gene":"POGZ","edge_type":0,"median_differential_logit":0.0315247029},{"source_gene":"C21ORF91","target_gene":"NUP54","edge_type":0,"median_differential_logit":0.0315262377},{"source_gene":"PAX6","target_gene":"ZNF124","edge_type":0,"median_differential_logit":0.0315253139},{"source_gene":"PPP2CA","target_gene":"VASP","edge_type":2,"median_differential_logit":0.0315300971},{"source_gene":"PAX6","target_gene":"SIX3","edge_type":1,"median_differential_logit":0.031532988},{"source_gene":"ZIC1","target_gene":"HRASLS","edge_type":0,"median_differential_logit":0.0315271169},{"source_gene":"PRKD3","target_gene":"PAK4","edge_type":1,"median_differential_logit":0.0315372795},{"source_gene":"LRP6","target_gene":"AXIN1","edge_type":1,"median_differential_logit":0.0315262079},{"source_gene":"PPP2CA","target_gene":"MKNK1","edge_type":2,"median_differential_logit":0.0315340757},{"source_gene":"NCK2","target_gene":"SF3B4","edge_type":0,"median_differential_logit":0.0315292329},{"source_gene":"PAX6","target_gene":"SET","edge_type":0,"median_differential_logit":0.0315208584},{"source_gene":"MYC","target_gene":"TFRC","edge_type":1,"median_differential_logit":0.0315274596},{"source_gene":"EGFR","target_gene":"PTK2","edge_type":1,"median_differential_logit":0.0315210074},{"source_gene":"USP1","target_gene":"FANCI","edge_type":1,"median_differential_logit":0.0315263867},{"source_gene":"E2F3","target_gene":"TFDP1","edge_type":1,"median_differential_logit":0.0315261632},{"source_gene":"DAZAP2","target_gene":"SOX10","edge_type":0,"median_differential_logit":-0.031525597},{"source_gene":"MPHOSPH10","target_gene":"BYSL","edge_type":0,"median_differential_logit":0.0315256119},{"source_gene":"PLK1","target_gene":"OPTN","edge_type":1,"median_differential_logit":0.0315243602},{"source_gene":"SOX10","target_gene":"DAZAP2","edge_type":0,"median_differential_logit":-0.0315255672},{"source_gene":"MSH2","target_gene":"BLM","edge_type":1,"median_differential_logit":0.0315254927},{"source_gene":"PSMB4","target_gene":"PFDN5","edge_type":0,"median_differential_logit":-0.0315255523},{"source_gene":"TP53BP2","target_gene":"LMNB2","edge_type":0,"median_differential_logit":0.0315255523},{"source_gene":"PFDN5","target_gene":"PSMB4","edge_type":0,"median_differential_logit":-0.0315255374},{"source_gene":"PFDN5","target_gene":"SCNM1","edge_type":0,"median_differential_logit":-0.0315255374},{"source_gene":"PFDN2","target_gene":"PFDN5","edge_type":0,"median_differential_logit":-0.0315255225},{"source_gene":"PFDN5","target_gene":"PFDN2","edge_type":0,"median_differential_logit":-0.0315255225},{"source_gene":"E2F1","target_gene":"TFDP1","edge_type":1,"median_differential_logit":0.0315197408},{"source_gene":"SCNM1","target_gene":"PFDN5","edge_type":0,"median_differential_logit":-0.0315255225},{"source_gene":"TIMM44","target_gene":"SFT2D2","edge_type":0,"median_differential_logit":0.0315255076},{"source_gene":"DAZAP2","target_gene":"LASP1","edge_type":0,"median_differential_logit":0.0315255523},{"source_gene":"CSNK2B","target_gene":"FAM64A","edge_type":0,"median_differential_logit":0.0315255523},{"source_gene":"CHEK2","target_gene":"FOXM1","edge_type":1,"median_differential_logit":0.0315257013},{"source_gene":"MSH6","target_gene":"BLM","edge_type":1,"median_differential_logit":0.0315256417},{"source_gene":"AGR2","target_gene":"CYB5R2","edge_type":0,"median_differential_logit":-0.0315254778},{"source_gene":"PFDN5","target_gene":"VGLL1","edge_type":0,"median_differential_logit":-0.0315254778},{"source_gene":"VGLL1","target_gene":"PFDN5","edge_type":0,"median_differential_logit":-0.0315254778},{"source_gene":"CCNC","target_gene":"ZNF688","edge_type":0,"median_differential_logit":-0.0315254778},{"source_gene":"ZNF688","target_gene":"CCNC","edge_type":0,"median_differential_logit":-0.0315254778},{"source_gene":"CYB5R2","target_gene":"AGR2","edge_type":0,"median_differential_logit":-0.0315254629},{"source_gene":"SUOX","target_gene":"CDCA4","edge_type":0,"median_differential_logit":-0.0315254629},{"source_gene":"LMO4","target_gene":"TRIM27","edge_type":0,"median_differential_logit":0.0315250158},{"source_gene":"CHEK2","target_gene":"AURKB","edge_type":1,"median_differential_logit":0.0315250158},{"source_gene":"CHEK2","target_gene":"STRAP","edge_type":1,"median_differential_logit":0.0315254182},{"source_gene":"RB1","target_gene":"E2F3","edge_type":2,"median_differential_logit":-0.0315254629},{"source_gene":"CDCA4","target_gene":"SUOX","edge_type":0,"median_differential_logit":-0.0315254331},{"source_gene":"SET","target_gene":"PAX6","edge_type":0,"median_differential_logit":0.031525448},{"source_gene":"BYSL","target_gene":"BHLHE40","edge_type":0,"median_differential_logit":-0.0315254331},{"source_gene":"LASP1","target_gene":"DAZAP2","edge_type":0,"median_differential_logit":0.031525448},{"source_gene":"CCNC","target_gene":"NOTCH1","edge_type":2,"median_differential_logit":0.0315253735},{"source_gene":"BHLHE40","target_gene":"BYSL","edge_type":0,"median_differential_logit":-0.0315254033},{"source_gene":"CYB5R2","target_gene":"SUOX","edge_type":0,"median_differential_logit":-0.0315253735},{"source_gene":"SUOX","target_gene":"CYB5R2","edge_type":0,"median_differential_logit":-0.0315253735},{"source_gene":"LMNB2","target_gene":"TP53BP2","edge_type":0,"median_differential_logit":0.0315254331},{"source_gene":"MDC1","target_gene":"RNF8","edge_type":1,"median_differential_logit":0.0315254629},{"source_gene":"TMEM14B","target_gene":"SLC16A10","edge_type":0,"median_differential_logit":0.0315253437},{"source_gene":"TRIM27","target_gene":"LMO4","edge_type":0,"median_differential_logit":0.0315254033},{"source_gene":"GOLGA2","target_gene":"CYB5R2","edge_type":0,"median_differential_logit":-0.0315253735},{"source_gene":"TEAD4","target_gene":"VGLL1","edge_type":0,"median_differential_logit":0.0315254629},{"source_gene":"CYB5R2","target_gene":"GOLGA2","edge_type":0,"median_differential_logit":-0.0315253437},{"source_gene":"BIRC2","target_gene":"RUNX1T1","edge_type":0,"median_differential_logit":-0.0315253437},{"source_gene":"EZH2","target_gene":"CDKN2A","edge_type":2,"median_differential_logit":0.0315250158},{"source_gene":"TP53BP2","target_gene":"CDC5L","edge_type":0,"median_differential_logit":0.0315252542},{"source_gene":"RUNX1T1","target_gene":"BIRC2","edge_type":0,"median_differential_logit":-0.0315253288},{"source_gene":"CDC25A","target_gene":"CDK6","edge_type":1,"median_differential_logit":0.0315257609},{"source_gene":"APH1A","target_gene":"IGFBP5","edge_type":0,"median_differential_logit":-0.0315253288},{"source_gene":"SLC16A10","target_gene":"TMEM14B","edge_type":0,"median_differential_logit":0.0315252692},{"source_gene":"AURKA","target_gene":"PARD3","edge_type":1,"median_differential_logit":0.0315252841},{"source_gene":"IGFBP5","target_gene":"ABHD4","edge_type":0,"median_differential_logit":0.0315253735},{"source_gene":"IGFBP5","target_gene":"APH1A","edge_type":0,"median_differential_logit":-0.0315253139},{"source_gene":"VEGFA","target_gene":"LCN2","edge_type":0,"median_differential_logit":0.0315253437},{"source_gene":"AURKB","target_gene":"PLEKHG6","edge_type":1,"median_differential_logit":0.0315254182},{"source_gene":"HGS","target_gene":"KRT18","edge_type":0,"median_differential_logit":-0.0315252841},{"source_gene":"KRT18","target_gene":"HGS","edge_type":0,"median_differential_logit":-0.0315252841},{"source_gene":"FAM90A1","target_gene":"CEP76","edge_type":0,"median_differential_logit":0.0315253139},{"source_gene":"CEP76","target_gene":"FAM90A1","edge_type":0,"median_differential_logit":0.031525299},{"source_gene":"TCEA2","target_gene":"TRAF2","edge_type":0,"median_differential_logit":-0.0315253139},{"source_gene":"TRAF2","target_gene":"TCEA2","edge_type":0,"median_differential_logit":-0.0315253139},{"source_gene":"ABHD4","target_gene":"IGFBP5","edge_type":0,"median_differential_logit":0.031525299},{"source_gene":"TMEM14B","target_gene":"SAR1B","edge_type":0,"median_differential_logit":-0.0315252841},{"source_gene":"ATAD2","target_gene":"ESR1","edge_type":1,"median_differential_logit":-0.0315252542},{"source_gene":"CDK2","target_gene":"ESR1","edge_type":1,"median_differential_logit":-0.0315252095},{"source_gene":"KCNK5","target_gene":"IGFBP5","edge_type":0,"median_differential_logit":-0.031525299},{"source_gene":"PDE9A","target_gene":"TRIM32","edge_type":0,"median_differential_logit":-0.0315252841},{"source_gene":"TRIM32","target_gene":"PDE9A","edge_type":0,"median_differential_logit":-0.0315252841},{"source_gene":"E2F3","target_gene":"CCNB1","edge_type":1,"median_differential_logit":0.0315252393},{"source_gene":"SAR1B","target_gene":"TMEM14B","edge_type":0,"median_differential_logit":-0.0315252542},{"source_gene":"IGFBP5","target_gene":"KCNK5","edge_type":0,"median_differential_logit":-0.0315252841},{"source_gene":"RNMT","target_gene":"KPNA2","edge_type":0,"median_differential_logit":0.0315253139},{"source_gene":"RINT1","target_gene":"SUOX","edge_type":0,"median_differential_logit":-0.0315251648},{"source_gene":"AURKB","target_gene":"RB1","edge_type":1,"median_differential_logit":-0.0315252692},{"source_gene":"ABHD4","target_gene":"REEP5","edge_type":0,"median_differential_logit":0.0315252095},{"source_gene":"TBX19","target_gene":"ZIC1","edge_type":0,"median_differential_logit":0.0315248668},{"source_gene":"HHEX","target_gene":"TP53BP2","edge_type":0,"median_differential_logit":-0.0315252841},{"source_gene":"SUOX","target_gene":"RINT1","edge_type":0,"median_differential_logit":-0.0315251499},{"source_gene":"CDC5L","target_gene":"TP53BP2","edge_type":0,"median_differential_logit":0.0315252692},{"source_gene":"TMEM14B","target_gene":"SFT2D2","edge_type":0,"median_differential_logit":0.0315249711},{"source_gene":"TP53BP2","target_gene":"HHEX","edge_type":0,"median_differential_logit":-0.0315252542},{"source_gene":"CSNK1A1","target_gene":"LRP6","edge_type":1,"median_differential_logit":-0.0315252244},{"source_gene":"NAA10","target_gene":"CASP8AP2","edge_type":0,"median_differential_logit":0.0315253735},{"source_gene":"LASP1","target_gene":"NUTF2","edge_type":0,"median_differential_logit":-0.0315251946},{"source_gene":"CDK1","target_gene":"USP1","edge_type":1,"median_differential_logit":0.0315251797},{"source_gene":"NUTF2","target_gene":"LASP1","edge_type":0,"median_differential_logit":-0.0315251797},{"source_gene":"CDK18","target_gene":"RABEP1","edge_type":0,"median_differential_logit":-0.0315252244},{"source_gene":"RABEP1","target_gene":"CDK18","edge_type":0,"median_differential_logit":-0.0315252244},{"source_gene":"HGS","target_gene":"LASP1","edge_type":0,"median_differential_logit":-0.0315251052},{"source_gene":"LASP1","target_gene":"HGS","edge_type":0,"median_differential_logit":-0.0315251052},{"source_gene":"AES","target_gene":"PSMB4","edge_type":0,"median_differential_logit":-0.031525135},{"source_gene":"PSMB4","target_gene":"AES","edge_type":0,"median_differential_logit":-0.031525135},{"source_gene":"MAPK1","target_gene":"AR","edge_type":1,"median_differential_logit":-0.0315252244},{"source_gene":"LASP1","target_gene":"YES1","edge_type":0,"median_differential_logit":-0.0315251946},{"source_gene":"YES1","target_gene":"LASP1","edge_type":0,"median_differential_logit":-0.0315251946},{"source_gene":"PAX6","target_gene":"CKS1B","edge_type":0,"median_differential_logit":0.0315252244},{"source_gene":"LASP1","target_gene":"CHERP","edge_type":0,"median_differential_logit":-0.0315252244},{"source_gene":"CCHCR1","target_gene":"CCDC53","edge_type":0,"median_differential_logit":-0.0315252244},{"source_gene":"CHERP","target_gene":"LASP1","edge_type":0,"median_differential_logit":-0.0315252244},{"source_gene":"NOTCH1","target_gene":"HES1","edge_type":1,"median_differential_logit":0.0315272212},{"source_gene":"CRY2","target_gene":"PDE9A","edge_type":0,"median_differential_logit":-0.0315250456},{"source_gene":"PDE9A","target_gene":"CRY2","edge_type":0,"median_differential_logit":-0.0315250456},{"source_gene":"TENC1","target_gene":"LASP1","edge_type":0,"median_differential_logit":0.0315252692},{"source_gene":"SCNM1","target_gene":"NAP1L2","edge_type":0,"median_differential_logit":-0.0315251648},{"source_gene":"NAP1L2","target_gene":"SCNM1","edge_type":0,"median_differential_logit":-0.0315251499},{"source_gene":"CCDC53","target_gene":"CCHCR1","edge_type":0,"median_differential_logit":-0.0315251946},{"source_gene":"REEP5","target_gene":"ABHD4","edge_type":0,"median_differential_logit":0.0315250903},{"source_gene":"PRPF3","target_gene":"RABEP1","edge_type":0,"median_differential_logit":-0.031525135},{"source_gene":"MICALL2","target_gene":"ACTN4","edge_type":0,"median_differential_logit":-0.0315251946},{"source_gene":"NUP54","target_gene":"CCHCR1","edge_type":0,"median_differential_logit":0.0315252244},{"source_gene":"CSNK1G3","target_gene":"LRP6","edge_type":1,"median_differential_logit":-0.031525135},{"source_gene":"C1ORF109","target_gene":"VGLL1","edge_type":0,"median_differential_logit":0.0315251946},{"source_gene":"RABEP1","target_gene":"PRPF3","edge_type":0,"median_differential_logit":-0.031525135},{"source_gene":"ACTN4","target_gene":"MICALL2","edge_type":0,"median_differential_logit":-0.0315251648},{"source_gene":"TFG","target_gene":"TBX19","edge_type":0,"median_differential_logit":0.0315251946},{"source_gene":"CCHCR1","target_gene":"NUP54","edge_type":0,"median_differential_logit":0.0315251946},{"source_gene":"BYSL","target_gene":"VIM","edge_type":0,"median_differential_logit":0.0315245688},{"source_gene":"TBK1","target_gene":"SIKE1","edge_type":1,"median_differential_logit":-0.0315250605},{"source_gene":"CHCHD3","target_gene":"CEP72","edge_type":0,"median_differential_logit":0.0315251648},{"source_gene":"CKS1B","target_gene":"PAX6","edge_type":0,"median_differential_logit":0.0315250903},{"source_gene":"SNRPA","target_gene":"AES","edge_type":0,"median_differential_logit":-0.0315250754},{"source_gene":"LHFPL2","target_gene":"FXYD6","edge_type":0,"median_differential_logit":-0.0315251648},{"source_gene":"ZNF318","target_gene":"AR","edge_type":2,"median_differential_logit":-0.031525135},{"source_gene":"ATAD2","target_gene":"AR","edge_type":1,"median_differential_logit":-0.0315250754},{"source_gene":"FXYD6","target_gene":"LHFPL2","edge_type":0,"median_differential_logit":-0.0315251648},{"source_gene":"AKT1","target_gene":"EZH2","edge_type":2,"median_differential_logit":-0.0315249562},{"source_gene":"AURKB","target_gene":"INCENP","edge_type":1,"median_differential_logit":0.0315250754},{"source_gene":"TRAF2","target_gene":"YES1","edge_type":0,"median_differential_logit":0.0315251201},{"source_gene":"SCNM1","target_gene":"TRIM27","edge_type":0,"median_differential_logit":0.0315251946},{"source_gene":"CCHCR1","target_gene":"ATG5","edge_type":0,"median_differential_logit":0.0315251946},{"source_gene":"MID1","target_gene":"UBE2L3","edge_type":0,"median_differential_logit":0.0315292776},{"source_gene":"AES","target_gene":"SNRPA","edge_type":0,"median_differential_logit":-0.0315250605},{"source_gene":"CHCHD3","target_gene":"PPP1CB","edge_type":0,"median_differential_logit":0.0315249264},{"source_gene":"TRIM27","target_gene":"SCNM1","edge_type":0,"median_differential_logit":0.0315251946},{"source_gene":"ATG5","target_gene":"CCHCR1","edge_type":0,"median_differential_logit":0.0315252095},{"source_gene":"CEP72","target_gene":"CHCHD3","edge_type":0,"median_differential_logit":0.0315250605},{"source_gene":"EGFR","target_gene":"GRB2","edge_type":1,"median_differential_logit":0.0315310359},{"source_gene":"FOXM1","target_gene":"CCND1","edge_type":1,"median_differential_logit":-0.0315250456},{"source_gene":"RNF8","target_gene":"CLK2","edge_type":0,"median_differential_logit":0.031524986},{"source_gene":"MAPK1","target_gene":"PCBP2","edge_type":1,"median_differential_logit":-0.0315250754},{"source_gene":"CLK2","target_gene":"RNF8","edge_type":0,"median_differential_logit":0.0315250307},{"source_gene":"CEP76","target_gene":"SUOX","edge_type":0,"median_differential_logit":-0.0315250009},{"source_gene":"SUOX","target_gene":"CEP76","edge_type":0,"median_differential_logit":-0.0315250158},{"source_gene":"KDM1A","target_gene":"AES","edge_type":0,"median_differential_logit":-0.0315249562},{"source_gene":"AGR2","target_gene":"NUP54","edge_type":0,"median_differential_logit":-0.0315249264},{"source_gene":"NUP54","target_gene":"AGR2","edge_type":0,"median_differential_logit":-0.0315249264},{"source_gene":"AES","target_gene":"KDM1A","edge_type":0,"median_differential_logit":-0.0315249413},{"source_gene":"VIM","target_gene":"BYSL","edge_type":0,"median_differential_logit":0.0315248817},{"source_gene":"S100A1","target_gene":"S100B","edge_type":0,"median_differential_logit":0.0315248221},{"source_gene":"GOLGA2","target_gene":"LMNB2","edge_type":0,"median_differential_logit":-0.0315249562},{"source_gene":"LMNB2","target_gene":"GOLGA2","edge_type":0,"median_differential_logit":-0.0315249562},{"source_gene":"PLK1","target_gene":"CHEK2","edge_type":1,"median_differential_logit":0.0315226018},{"source_gene":"FAM90A1","target_gene":"PFDN5","edge_type":0,"median_differential_logit":-0.0315250456},{"source_gene":"PFDN5","target_gene":"FAM90A1","edge_type":0,"median_differential_logit":-0.0315250605},{"source_gene":"PFDN5","target_gene":"KCTD9","edge_type":0,"median_differential_logit":-0.0315250009},{"source_gene":"KRT4","target_gene":"KRT16","edge_type":0,"median_differential_logit":0.0315250158},{"source_gene":"KCTD9","target_gene":"PFDN5","edge_type":0,"median_differential_logit":-0.0315250009},{"source_gene":"DTNB","target_gene":"DMD","edge_type":0,"median_differential_logit":0.0315105319},{"source_gene":"LASP1","target_gene":"TRIP13","edge_type":0,"median_differential_logit":-0.031524688},{"source_gene":"TRIP13","target_gene":"LASP1","edge_type":0,"median_differential_logit":-0.0315246731},{"source_gene":"FAM90A1","target_gene":"TRAF2","edge_type":0,"median_differential_logit":0.0315250456},{"source_gene":"TRAF2","target_gene":"FAM90A1","edge_type":0,"median_differential_logit":0.0315249711}],"hallmark":[{"panel":"strict_5of5::Basal-like","collection":"Hallmark","term":"HALLMARK_MYC_TARGETS_V1","panel_size":257,"set_size":147,"overlap_n":13,"fold_enrichment":2.4538235528,"p_value":0.0023974026,"fdr":0.0199783551,"meets_min_overlap":true,"min_overlap":2,"n_tests":50,"overlap_genes":"CDK2;CUL1;HNRNPC;KPNA2;MYC;ODC1;PCNA;RAD23B;SET;SNRPA;TFDP1;UBE2L3;USP1"},{"panel":"strict_5of5::Basal-like","collection":"Hallmark","term":"HALLMARK_PI3K_AKT_MTOR_SIGNALING","panel_size":257,"set_size":94,"overlap_n":12,"fold_enrichment":3.5421806441,"p_value":0.0001276057,"fdr":0.0021267622,"meets_min_overlap":true,"min_overlap":2,"n_tests":50,"overlap_genes":"AKT1;CDK1;CDK2;CSNK2B;E2F1;EGFR;GRB2;MAPK1;MKNK1;PAK4;TBK1;TRAF2"},{"panel":"strict_5of5::Basal-like","collection":"Hallmark","term":"HALLMARK_E2F_TARGETS","panel_size":257,"set_size":153,"overlap_n":20,"fold_enrichment":3.6270695048,"p_value":4.926e-07,"fdr":1.23138e-05,"meets_min_overlap":true,"min_overlap":2,"n_tests":50,"overlap_genes":"ATAD2;AURKA;AURKB;CDC25A;CDK1;CDKN2A;CHEK2;CKS1B;EZH2;HMGB3;KPNA2;LMNB1;MSH2;MYC;PCNA;PLK1;TFRC;TK1;TRIP13;USP1"},{"panel":"strict_5of5::Basal-like","collection":"Hallmark","term":"HALLMARK_G2M_CHECKPOINT","panel_size":257,"set_size":167,"overlap_n":24,"fold_enrichment":3.9876045574,"p_value":5e-09,"fdr":2.505e-07,"meets_min_overlap":true,"min_overlap":2,"n_tests":50,"overlap_genes":"AURKA;AURKB;CASP8AP2;CCND1;CDC25A;CDK1;CHAF1A;CKS1B;CUL1;DMD;E2F1;E2F3;EFNA5;EZH2;HMGB3;INCENP;KPNA2;LMNB1;MYC;ODC1;PLK1;PML;RAD23B;TFDP1"},{"panel":"strict_5of5::Basal-like","collection":"Hallmark","term":"HALLMARK_WNT_BETA_CATENIN_SIGNALING","panel_size":257,"set_size":37,"overlap_n":6,"fold_enrichment":4.4995267641,"p_value":0.0018760426,"fdr":0.0187604259,"meets_min_overlap":true,"min_overlap":2,"n_tests":50,"overlap_genes":"AXIN1;CUL1;MYC;NOTCH1;SKP2;WNT5B"},{"panel":"strict_5of5::Basal-like","collection":"Hallmark","term":"HALLMARK_NOTCH_SIGNALING","panel_size":257,"set_size":27,"overlap_n":6,"fold_enrichment":6.1660181582,"p_value":0.0003230909,"fdr":0.0040386356,"meets_min_overlap":true,"min_overlap":2,"n_tests":50,"overlap_genes":"APH1A;CCND1;CUL1;FZD7;HES1;NOTCH1"}],"matched":[{"graph_class":"LumA","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.3127753304,"null_mean":0.2135462555,"null_sd":0.0264667428,"fold_over_null":1.4646725116,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.005994006,"fdr_global":0.0095904096},{"graph_class":"LumA","term":"Basal-like","statistic":"both_endpoints","observed":0.0352422907,"null_mean":0.0153832599,"null_sd":0.0079039666,"fold_over_null":2.2909507446,"empirical_p":0.022977023,"n_draws":1000,"fdr_within_panel":0.0719280719,"fdr_global":0.1046226501},{"graph_class":"LumA","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1176470588,"null_mean":0.104471053,"null_sd":0.0134458602,"fold_over_null":1.1261211165,"empirical_p":0.1638361638,"n_draws":1000,"fdr_within_panel":0.2457542458,"fdr_global":0.4368964369},{"graph_class":"LumA","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.2290748899,"null_mean":0.0658986784,"null_sd":0.016612977,"fold_over_null":3.4761681931,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.005994006,"fdr_global":0.0095904096},{"graph_class":"LumA","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0011674009,"null_sd":0.0022593495,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumA","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0452488688,"null_mean":0.0304088574,"null_sd":0.0078301313,"fold_over_null":1.4880160818,"empirical_p":0.035964036,"n_draws":1000,"fdr_within_panel":0.0799200799,"fdr_global":0.1370058513},{"graph_class":"LumA","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0220264317,"null_mean":0.0109911894,"null_sd":0.0065580958,"fold_over_null":2.004008016,"empirical_p":0.1028971029,"n_draws":1000,"fdr_within_panel":0.1763950335,"fdr_global":0.3086913087},{"graph_class":"LumA","term":"LumA","statistic":"both_endpoints","observed":0.0044052863,"null_mean":0.0001762115,"null_sd":0.0008858947,"fold_over_null":25.0,"empirical_p":0.03996004,"n_draws":1000,"fdr_within_panel":0.0799200799,"fdr_global":0.1370058513},{"graph_class":"LumA","term":"LumA","statistic":"unique_node_fraction","observed":0.0135746606,"null_mean":0.0062222095,"null_sd":0.0035779215,"fold_over_null":2.1816463461,"empirical_p":0.023976024,"n_draws":1000,"fdr_within_panel":0.0719280719,"fdr_global":0.1046226501},{"graph_class":"LumA","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0,"null_mean":0.0195814978,"null_sd":0.0089754681,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumA","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":6.1674e-05,"null_sd":0.0005178379,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumA","term":"LumB","statistic":"unique_node_fraction","observed":0.0,"null_mean":0.0107863474,"null_sd":0.004771,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.2408163265,"null_mean":0.1888122449,"null_sd":0.0248238804,"fold_over_null":1.2754274844,"empirical_p":0.023976024,"n_draws":1000,"fdr_within_panel":0.2877122877,"fdr_global":0.1046226501},{"graph_class":"LumB","term":"Basal-like","statistic":"both_endpoints","observed":0.0,"null_mean":0.011244898,"null_sd":0.0065453234,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1146788991,"null_mean":0.0980114103,"null_sd":0.0128386369,"fold_over_null":1.1700566166,"empirical_p":0.1008991009,"n_draws":1000,"fdr_within_panel":0.6053946054,"fdr_global":0.3086913087},{"graph_class":"LumB","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0408163265,"null_mean":0.0590285714,"null_sd":0.0146013831,"fold_over_null":0.6914672936,"empirical_p":0.9340659341,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0014285714,"null_sd":0.0024416825,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0229357798,"null_mean":0.0310546188,"null_sd":0.0071886459,"fold_over_null":0.7385625931,"empirical_p":0.8741258741,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0,"null_mean":0.0102367347,"null_sd":0.0062280208,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001142857,"null_sd":0.000673695,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"LumA","statistic":"unique_node_fraction","observed":0.0,"null_mean":0.0057858933,"null_sd":0.0033368281,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0081632653,"null_mean":0.0185755102,"null_sd":0.0082318208,"fold_over_null":0.4394638541,"empirical_p":0.956043956,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001877551,"null_sd":0.0008747457,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"LumB","term":"LumB","statistic":"unique_node_fraction","observed":0.0091743119,"null_mean":0.0103350364,"null_sd":0.0043257062,"fold_over_null":0.8876903344,"empirical_p":0.6123876124,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"HER2-enriched","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.1145833333,"null_mean":0.1854965278,"null_sd":0.0221697617,"fold_over_null":0.6177114726,"empirical_p":0.999000999,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"HER2-enriched","term":"Basal-like","statistic":"both_endpoints","observed":0.0034722222,"null_mean":0.0101354167,"null_sd":0.0058093487,"fold_over_null":0.3425830764,"empirical_p":0.954045954,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"HER2-enriched","term":"Basal-like","statistic":"unique_node_fraction","observed":0.06,"null_mean":0.0969985209,"null_sd":0.0113566762,"fold_over_null":0.6185661335,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"HER2-enriched","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0763888889,"null_mean":0.0623993056,"null_sd":0.0137615109,"fold_over_null":1.2241945356,"empirical_p":0.1868131868,"n_draws":1000,"fdr_within_panel":0.5604395604,"fdr_global":0.4719491035},{"graph_class":"HER2-enriched","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0014652778,"null_sd":0.0022023808,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"HER2-enriched","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0433333333,"null_mean":0.031695725,"null_sd":0.0064687692,"fold_over_null":1.3671664991,"empirical_p":0.037962038,"n_draws":1000,"fdr_within_panel":0.2277722278,"fdr_global":0.1370058513},{"graph_class":"HER2-enriched","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0173611111,"null_mean":0.0118611111,"null_sd":0.0062713814,"fold_over_null":1.4637002342,"empirical_p":0.2587412587,"n_draws":1000,"fdr_within_panel":0.620979021,"fdr_global":0.620979021},{"graph_class":"HER2-enriched","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":7.63889e-05,"null_sd":0.000509572,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"HER2-enriched","term":"LumA","statistic":"unique_node_fraction","observed":0.0133333333,"null_mean":0.0060802832,"null_sd":0.0030562864,"fold_over_null":2.1928803206,"empirical_p":0.013986014,"n_draws":1000,"fdr_within_panel":0.1678321678,"fdr_global":0.0959040959},{"graph_class":"HER2-enriched","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0208333333,"null_mean":0.0177013889,"null_sd":0.0077074913,"fold_over_null":1.1769321302,"empirical_p":0.3876123876,"n_draws":1000,"fdr_within_panel":0.7752247752,"fdr_global":0.8859711717},{"graph_class":"HER2-enriched","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":7.29167e-05,"null_sd":0.0004981106,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"HER2-enriched","term":"LumB","statistic":"unique_node_fraction","observed":0.0133333333,"null_mean":0.0095513733,"null_sd":0.0038964118,"fold_over_null":1.3959598207,"empirical_p":0.1578421578,"n_draws":1000,"fdr_within_panel":0.5604395604,"fdr_global":0.4368964369},{"graph_class":"Basal-like","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.4154929577,"null_mean":0.1885633803,"null_sd":0.0223450292,"fold_over_null":2.2034657903,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.0095904096},{"graph_class":"Basal-like","term":"Basal-like","statistic":"both_endpoints","observed":0.0774647887,"null_mean":0.0116619718,"null_sd":0.0062332793,"fold_over_null":6.6425120773,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.0095904096},{"graph_class":"Basal-like","term":"Basal-like","statistic":"unique_node_fraction","observed":0.2412451362,"null_mean":0.0978262227,"null_sd":0.0113719599,"fold_over_null":2.4660579707,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.0095904096},{"graph_class":"Basal-like","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0669014085,"null_mean":0.0649647887,"null_sd":0.0143609034,"fold_over_null":1.0298102981,"empirical_p":0.4625374625,"n_draws":1000,"fdr_within_panel":0.9250749251,"fdr_global":1.0},{"graph_class":"Basal-like","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0014929577,"null_sd":0.0022673395,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"Basal-like","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0311284047,"null_mean":0.0319067663,"null_sd":0.0065262549,"fold_over_null":0.9756051223,"empirical_p":0.5504495504,"n_draws":1000,"fdr_within_panel":0.9436278008,"fdr_global":1.0},{"graph_class":"Basal-like","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0,"null_mean":0.0101232394,"null_sd":0.0056950794,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"Basal-like","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":2.46479e-05,"null_sd":0.0002937126,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"Basal-like","term":"LumA","statistic":"unique_node_fraction","observed":0.0,"null_mean":0.0054553595,"null_sd":0.0029482904,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"Basal-like","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0422535211,"null_mean":0.0186091549,"null_sd":0.0078772412,"fold_over_null":2.270577105,"empirical_p":0.004995005,"n_draws":1000,"fdr_within_panel":0.014985015,"fdr_global":0.03996004},{"graph_class":"Basal-like","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001338028,"null_sd":0.000673563,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"graph_class":"Basal-like","term":"LumB","statistic":"unique_node_fraction","observed":0.0194552529,"null_mean":0.0101246376,"null_sd":0.0041381722,"fold_over_null":1.9215752347,"empirical_p":0.018981019,"n_draws":1000,"fdr_within_panel":0.0455544456,"fdr_global":0.1046226501}],"cross":[{"graph_class":"LumA","annotation_class":"LumA","node_fraction":0.0135746606,"focal_fraction":0.0135746606,"best_alternative_fraction":0.1176470588,"specificity_difference":-0.1040723982,"focal_is_largest":false},{"graph_class":"LumA","annotation_class":"LumB","node_fraction":0.0,"focal_fraction":0.0135746606,"best_alternative_fraction":0.1176470588,"specificity_difference":-0.1040723982,"focal_is_largest":false},{"graph_class":"LumA","annotation_class":"HER2-enriched","node_fraction":0.0452488688,"focal_fraction":0.0135746606,"best_alternative_fraction":0.1176470588,"specificity_difference":-0.1040723982,"focal_is_largest":false},{"graph_class":"LumA","annotation_class":"Basal-like","node_fraction":0.1176470588,"focal_fraction":0.0135746606,"best_alternative_fraction":0.1176470588,"specificity_difference":-0.1040723982,"focal_is_largest":false},{"graph_class":"LumB","annotation_class":"LumA","node_fraction":0.0,"focal_fraction":0.0091743119,"best_alternative_fraction":0.1146788991,"specificity_difference":-0.1055045872,"focal_is_largest":false},{"graph_class":"LumB","annotation_class":"LumB","node_fraction":0.0091743119,"focal_fraction":0.0091743119,"best_alternative_fraction":0.1146788991,"specificity_difference":-0.1055045872,"focal_is_largest":false},{"graph_class":"LumB","annotation_class":"HER2-enriched","node_fraction":0.0229357798,"focal_fraction":0.0091743119,"best_alternative_fraction":0.1146788991,"specificity_difference":-0.1055045872,"focal_is_largest":false},{"graph_class":"LumB","annotation_class":"Basal-like","node_fraction":0.1146788991,"focal_fraction":0.0091743119,"best_alternative_fraction":0.1146788991,"specificity_difference":-0.1055045872,"focal_is_largest":false},{"graph_class":"HER2-enriched","annotation_class":"LumA","node_fraction":0.0133333333,"focal_fraction":0.0433333333,"best_alternative_fraction":0.06,"specificity_difference":-0.0166666667,"focal_is_largest":false},{"graph_class":"HER2-enriched","annotation_class":"LumB","node_fraction":0.0133333333,"focal_fraction":0.0433333333,"best_alternative_fraction":0.06,"specificity_difference":-0.0166666667,"focal_is_largest":false},{"graph_class":"HER2-enriched","annotation_class":"HER2-enriched","node_fraction":0.0433333333,"focal_fraction":0.0433333333,"best_alternative_fraction":0.06,"specificity_difference":-0.0166666667,"focal_is_largest":false},{"graph_class":"HER2-enriched","annotation_class":"Basal-like","node_fraction":0.06,"focal_fraction":0.0433333333,"best_alternative_fraction":0.06,"specificity_difference":-0.0166666667,"focal_is_largest":false},{"graph_class":"Basal-like","annotation_class":"LumA","node_fraction":0.0,"focal_fraction":0.2412451362,"best_alternative_fraction":0.0311284047,"specificity_difference":0.2101167315,"focal_is_largest":true},{"graph_class":"Basal-like","annotation_class":"LumB","node_fraction":0.0194552529,"focal_fraction":0.2412451362,"best_alternative_fraction":0.0311284047,"specificity_difference":0.2101167315,"focal_is_largest":true},{"graph_class":"Basal-like","annotation_class":"HER2-enriched","node_fraction":0.0311284047,"focal_fraction":0.2412451362,"best_alternative_fraction":0.0311284047,"specificity_difference":0.2101167315,"focal_is_largest":true},{"graph_class":"Basal-like","annotation_class":"Basal-like","node_fraction":0.2412451362,"focal_fraction":0.2412451362,"best_alternative_fraction":0.0311284047,"specificity_difference":0.2101167315,"focal_is_largest":true}],"rewired":[{"graph_class":"Basal-like","observed_coherent_fraction":0.0812854442,"null_mean":0.0487296786,"null_sd":0.0074917446,"fold_over_null":1.6680890682,"empirical_p":0.001998002,"n_rewirings":1000,"median_accepted_swaps":5290.0,"median_changed_edge_rows":524.0,"fdr_four_graphs":0.003996004},{"graph_class":"HER2-enriched","observed_coherent_fraction":0.0586011342,"null_mean":0.0357599244,"null_sd":0.0072804463,"fold_over_null":1.6387376434,"empirical_p":0.002997003,"n_rewirings":1000,"median_accepted_swaps":5290.0,"median_changed_edge_rows":525.0,"fdr_four_graphs":0.003996004},{"graph_class":"LumA","observed_coherent_fraction":0.1606805293,"null_mean":0.0772835539,"null_sd":0.0094010059,"fold_over_null":2.079103784,"empirical_p":0.000999001,"n_rewirings":1000,"median_accepted_swaps":5290.0,"median_changed_edge_rows":524.0,"fdr_four_graphs":0.003996004},{"graph_class":"LumB","observed_coherent_fraction":0.0529300567,"null_mean":0.034536862,"null_sd":0.0072723323,"fold_over_null":1.5325670498,"empirical_p":0.010989011,"n_rewirings":1000,"median_accepted_swaps":5290.0,"median_changed_edge_rows":524.0,"fdr_four_graphs":0.010989011}],"external":[{"scope":"TCGA-BRCA","metric":"spearman_development_external","is_primary_metric":true,"observed":0.6114135852,"null_mean":0.5362843419,"null_sd":0.0454686593,"null_q025":0.4420180728,"null_q975":0.6170273692,"observed_minus_null_mean":0.0751292432,"standardized_difference":1.6523302951,"empirical_p_upper":0.038961039,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.038961039,"fdr_primary_four_scopes":0.038961039},{"scope":"TCGA-BRCA","metric":"sign_agreement_development_external","is_primary_metric":false,"observed":0.764084507,"null_mean":0.680834507,"null_sd":0.0270331038,"null_q025":0.6267605634,"null_q975":0.7323943662,"observed_minus_null_mean":0.08325,"standardized_difference":3.0795575899,"empirical_p_upper":0.000999001,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0014985015,"fdr_primary_four_scopes":null},{"scope":"TCGA-BRCA","metric":"sign_agreement_learned_external","is_primary_metric":false,"observed":0.7605633803,"null_mean":0.6310669014,"null_sd":0.0278596286,"null_q025":0.5738556338,"null_q975":0.6866197183,"observed_minus_null_mean":0.1294964789,"standardized_difference":4.6481767777,"empirical_p_upper":0.000999001,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0014985015,"fdr_primary_four_scopes":null},{"scope":"GSE21653","metric":"spearman_development_external","is_primary_metric":true,"observed":0.3734430954,"null_mean":0.2746906263,"null_sd":0.0561231937,"null_q025":0.1628505771,"null_q975":0.3755126646,"observed_minus_null_mean":0.0987524691,"standardized_difference":1.7595660995,"empirical_p_upper":0.028971029,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0316047589,"fdr_primary_four_scopes":0.0386280386},{"scope":"GSE21653","metric":"sign_agreement_development_external","is_primary_metric":false,"observed":0.676056338,"null_mean":0.5827957746,"null_sd":0.0292560586,"null_q025":0.5246478873,"null_q975":0.6408450704,"observed_minus_null_mean":0.0932605634,"standardized_difference":3.1877350456,"empirical_p_upper":0.000999001,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0014985015,"fdr_primary_four_scopes":null},{"scope":"GSE21653","metric":"sign_agreement_learned_external","is_primary_metric":false,"observed":0.6795774648,"null_mean":0.5644084507,"null_sd":0.029772627,"null_q025":0.5070422535,"null_q975":0.6232394366,"observed_minus_null_mean":0.1151690141,"standardized_difference":3.8682852505,"empirical_p_upper":0.001998002,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0026640027,"fdr_primary_four_scopes":null},{"scope":"GSE1456","metric":"spearman_development_external","is_primary_metric":true,"observed":0.4550511042,"null_mean":0.1815073376,"null_sd":0.0596352588,"null_q025":0.0607108442,"null_q975":0.2965097382,"observed_minus_null_mean":0.2735437666,"standardized_difference":4.5869469187,"empirical_p_upper":0.000999001,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0014985015,"fdr_primary_four_scopes":0.003996004},{"scope":"GSE1456","metric":"sign_agreement_development_external","is_primary_metric":false,"observed":0.7535211268,"null_mean":0.5591197183,"null_sd":0.0299163364,"null_q025":0.5035211268,"null_q975":0.6197183099,"observed_minus_null_mean":0.1944014085,"standardized_difference":6.498168964,"empirical_p_upper":0.000999001,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0014985015,"fdr_primary_four_scopes":null},{"scope":"GSE1456","metric":"sign_agreement_learned_external","is_primary_metric":false,"observed":0.75,"null_mean":0.5450492958,"null_sd":0.0293521458,"null_q025":0.4894366197,"null_q975":0.6056338028,"observed_minus_null_mean":0.2049507042,"standardized_difference":6.982477714,"empirical_p_upper":0.000999001,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0014985015,"fdr_primary_four_scopes":null},{"scope":"EXTERNAL_META","metric":"spearman_development_external","is_primary_metric":true,"observed":0.667557368,"null_mean":0.5563265677,"null_sd":0.0435657434,"null_q025":0.4699365328,"null_q975":0.6355506093,"observed_minus_null_mean":0.1112308003,"standardized_difference":2.5531711787,"empirical_p_upper":0.004995005,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.005994006,"fdr_primary_four_scopes":0.00999001},{"scope":"EXTERNAL_META","metric":"sign_agreement_development_external","is_primary_metric":false,"observed":0.8169014085,"null_mean":0.6894260563,"null_sd":0.026934273,"null_q025":0.6373239437,"null_q975":0.735915493,"observed_minus_null_mean":0.1274753521,"standardized_difference":4.7328306322,"empirical_p_upper":0.000999001,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0014985015,"fdr_primary_four_scopes":null},{"scope":"EXTERNAL_META","metric":"sign_agreement_learned_external","is_primary_metric":false,"observed":0.8133802817,"null_mean":0.6384190141,"null_sd":0.0279501374,"null_q025":0.5809859155,"null_q975":0.6971830986,"observed_minus_null_mean":0.1749612676,"standardized_difference":6.2597641351,"empirical_p_upper":0.000999001,"n_draws":1000,"matching":"edge_type+prior_sign+source_out_degree_bin+target_in_degree_bin","selected_edges_excluded_from_null":true,"fdr_all_12_tests":0.0014985015,"fdr_primary_four_scopes":null}],"ablation":[{"arm":"FULL","mean_pairwise_edge_jaccard":0.6039258141,"strict_edges":284,"strict_nodes":257,"strict_n_edges":284,"strict_n_nodes":257,"strict_n_components":52,"strict_giant_nodes":84,"strict_giant_fraction":0.326848249,"strict_dyad_components":19,"strict_degree_one_nodes":118,"consensus_n_edges":529,"consensus_n_nodes":480,"consensus_n_components":87,"consensus_giant_nodes":209,"consensus_giant_fraction":0.4354166667,"consensus_dyad_components":42,"consensus_degree_one_nodes":239,"strict_edge_jaccard_vs_full":1.0,"strict_node_jaccard_vs_full":1.0,"basal_like_unique_node_fraction":0.3035019455,"best_non_basal_annotation_fraction":0.0311284047,"basal_specificity_margin":0.2723735409,"basal_annotation_is_largest":true,"basal_observed__at_least_one_endpoint":0.5633802817,"basal_observed__both_endpoints":0.1126760563,"basal_observed__unique_node_fraction":0.3035019455,"basal_fold_over_null__at_least_one_endpoint":2.522585019,"basal_fold_over_null__both_endpoints":7.561436673,"basal_fold_over_null__unique_node_fraction":2.553694011,"basal_empirical_p__at_least_one_endpoint":0.000999001,"basal_empirical_p__both_endpoints":0.000999001,"basal_empirical_p__unique_node_fraction":0.000999001,"basal_fdr_global_15_tests__at_least_one_endpoint":0.0012487512,"basal_fdr_global_15_tests__both_endpoints":0.0012487512,"basal_fdr_global_15_tests__unique_node_fraction":0.0012487512,"n_significant_panel_fdr__C2_CGP":135.0,"n_significant_panel_fdr__Hallmark":6.0},{"arm":"NO_CONTEXT_LOSS","mean_pairwise_edge_jaccard":0.0771609505,"strict_edges":2,"strict_nodes":4,"strict_n_edges":2,"strict_n_nodes":4,"strict_n_components":2,"strict_giant_nodes":2,"strict_giant_fraction":0.5,"strict_dyad_components":2,"strict_degree_one_nodes":4,"consensus_n_edges":529,"consensus_n_nodes":562,"consensus_n_components":61,"consensus_giant_nodes":341,"consensus_giant_fraction":0.6067615658,"consensus_dyad_components":32,"consensus_degree_one_nodes":348,"strict_edge_jaccard_vs_full":0.0,"strict_node_jaccard_vs_full":0.0077220077,"basal_like_unique_node_fraction":0.25,"best_non_basal_annotation_fraction":0.0,"basal_specificity_margin":0.25,"basal_annotation_is_largest":true,"basal_observed__at_least_one_endpoint":0.5,"basal_observed__both_endpoints":0.0,"basal_observed__unique_node_fraction":0.25,"basal_fold_over_null__at_least_one_endpoint":1.9193857965,"basal_fold_over_null__both_endpoints":0.0,"basal_fold_over_null__unique_node_fraction":1.7361111111,"basal_empirical_p__at_least_one_endpoint":0.4525474525,"basal_empirical_p__both_endpoints":1.0,"basal_empirical_p__unique_node_fraction":0.4525474525,"basal_fdr_global_15_tests__at_least_one_endpoint":0.4848722706,"basal_fdr_global_15_tests__both_endpoints":1.0,"basal_fdr_global_15_tests__unique_node_fraction":0.4848722706,"n_significant_panel_fdr__C2_CGP":null,"n_significant_panel_fdr__Hallmark":null},{"arm":"NO_COHORT_QUERY_LOSS","mean_pairwise_edge_jaccard":0.5205171342,"strict_edges":239,"strict_nodes":171,"strict_n_edges":239,"strict_n_nodes":171,"strict_n_components":25,"strict_giant_nodes":65,"strict_giant_fraction":0.3801169591,"strict_dyad_components":7,"strict_degree_one_nodes":39,"consensus_n_edges":529,"consensus_n_nodes":375,"consensus_n_components":42,"consensus_giant_nodes":218,"consensus_giant_fraction":0.5813333333,"consensus_dyad_components":9,"consensus_degree_one_nodes":110,"strict_edge_jaccard_vs_full":0.4815864023,"strict_node_jaccard_vs_full":0.4032786885,"basal_like_unique_node_fraction":0.3040935673,"best_non_basal_annotation_fraction":0.0233918129,"basal_specificity_margin":0.2807017544,"basal_annotation_is_largest":true,"basal_observed__at_least_one_endpoint":0.5606694561,"basal_observed__both_endpoints":0.1046025105,"basal_observed__unique_node_fraction":0.3040935673,"basal_fold_over_null__at_least_one_endpoint":2.7075082842,"basal_fold_over_null__both_endpoints":7.6312576313,"basal_fold_over_null__unique_node_fraction":2.6627576181,"basal_empirical_p__at_least_one_endpoint":0.000999001,"basal_empirical_p__both_endpoints":0.000999001,"basal_empirical_p__unique_node_fraction":0.000999001,"basal_fdr_global_15_tests__at_least_one_endpoint":0.0012487512,"basal_fdr_global_15_tests__both_endpoints":0.0012487512,"basal_fdr_global_15_tests__unique_node_fraction":0.0012487512,"n_significant_panel_fdr__C2_CGP":73.0,"n_significant_panel_fdr__Hallmark":4.0},{"arm":"NO_EDGE_TYPE_CALIBRATION","mean_pairwise_edge_jaccard":0.5441157999,"strict_edges":264,"strict_nodes":192,"strict_n_edges":264,"strict_n_nodes":192,"strict_n_components":27,"strict_giant_nodes":106,"strict_giant_fraction":0.5520833333,"strict_dyad_components":4,"strict_degree_one_nodes":45,"consensus_n_edges":529,"consensus_n_nodes":384,"consensus_n_components":40,"consensus_giant_nodes":246,"consensus_giant_fraction":0.640625,"consensus_dyad_components":11,"consensus_degree_one_nodes":136,"strict_edge_jaccard_vs_full":0.4691689008,"strict_node_jaccard_vs_full":0.3987538941,"basal_like_unique_node_fraction":0.2864583333,"best_non_basal_annotation_fraction":0.0260416667,"basal_specificity_margin":0.2604166667,"basal_annotation_is_largest":true,"basal_observed__at_least_one_endpoint":0.5454545455,"basal_observed__both_endpoints":0.0984848485,"basal_observed__unique_node_fraction":0.2864583333,"basal_fold_over_null__at_least_one_endpoint":2.5396825397,"basal_fold_over_null__both_endpoints":6.6906845085,"basal_fold_over_null__unique_node_fraction":2.4526004638,"basal_empirical_p__at_least_one_endpoint":0.000999001,"basal_empirical_p__both_endpoints":0.000999001,"basal_empirical_p__unique_node_fraction":0.000999001,"basal_fdr_global_15_tests__at_least_one_endpoint":0.0012487512,"basal_fdr_global_15_tests__both_endpoints":0.0012487512,"basal_fdr_global_15_tests__unique_node_fraction":0.0012487512,"n_significant_panel_fdr__C2_CGP":87.0,"n_significant_panel_fdr__Hallmark":4.0},{"arm":"NO_SIGNED_WEIGHT","mean_pairwise_edge_jaccard":0.561450784,"strict_edges":272,"strict_nodes":229,"strict_n_edges":272,"strict_n_nodes":229,"strict_n_components":41,"strict_giant_nodes":71,"strict_giant_fraction":0.3100436681,"strict_dyad_components":11,"strict_degree_one_nodes":88,"consensus_n_edges":529,"consensus_n_nodes":466,"consensus_n_components":78,"consensus_giant_nodes":244,"consensus_giant_fraction":0.5236051502,"consensus_dyad_components":41,"consensus_degree_one_nodes":228,"strict_edge_jaccard_vs_full":0.6401179941,"strict_node_jaccard_vs_full":0.6254180602,"basal_like_unique_node_fraction":0.3318777293,"best_non_basal_annotation_fraction":0.0305676856,"basal_specificity_margin":0.3013100437,"basal_annotation_is_largest":true,"basal_observed__at_least_one_endpoint":0.5772058824,"basal_observed__both_endpoints":0.1323529412,"basal_observed__unique_node_fraction":0.3318777293,"basal_fold_over_null__at_least_one_endpoint":2.6685250025,"basal_fold_over_null__both_endpoints":8.7569934323,"basal_fold_over_null__unique_node_fraction":2.8453281218,"basal_empirical_p__at_least_one_endpoint":0.000999001,"basal_empirical_p__both_endpoints":0.000999001,"basal_empirical_p__unique_node_fraction":0.000999001,"basal_fdr_global_15_tests__at_least_one_endpoint":0.0012487512,"basal_fdr_global_15_tests__both_endpoints":0.0012487512,"basal_fdr_global_15_tests__unique_node_fraction":0.0012487512,"n_significant_panel_fdr__C2_CGP":128.0,"n_significant_panel_fdr__Hallmark":5.0}],"ablation_tests":[{"panel":"FULL","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.4154929577,"null_mean":0.1889295775,"null_sd":0.0222086387,"fold_over_null":2.199194871,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"FULL","term":"Basal-like","statistic":"both_endpoints","observed":0.0774647887,"null_mean":0.0114084507,"null_sd":0.0062429722,"fold_over_null":6.7901234568,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"FULL","term":"Basal-like","statistic":"unique_node_fraction","observed":0.2412451362,"null_mean":0.0980776191,"null_sd":0.0114535806,"fold_over_null":2.4597368743,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_COHORT_QUERY_LOSS","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.3640167364,"null_mean":0.1633472803,"null_sd":0.0239435662,"fold_over_null":2.2284836066,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_COHORT_QUERY_LOSS","term":"Basal-like","statistic":"both_endpoints","observed":0.0627615063,"null_mean":0.009,"null_sd":0.0062291341,"fold_over_null":6.9735006974,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_COHORT_QUERY_LOSS","term":"Basal-like","statistic":"unique_node_fraction","observed":0.216374269,"null_mean":0.0887843046,"null_sd":0.0128302618,"fold_over_null":2.4370779273,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_CONTEXT_LOSS","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.5,"null_mean":0.2945,"null_sd":0.331103276,"fold_over_null":1.6977928693,"empirical_p":0.4915084915,"n_draws":1000,"fdr_within_panel":0.7372627373,"fdr_global":0.5266162409},{"panel":"NO_CONTEXT_LOSS","term":"Basal-like","statistic":"both_endpoints","observed":0.0,"null_mean":0.036,"null_sd":0.1349897115,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"NO_CONTEXT_LOSS","term":"Basal-like","statistic":"unique_node_fraction","observed":0.25,"null_mean":0.1654166667,"null_sd":0.1978427302,"fold_over_null":1.5113350126,"empirical_p":0.4915084915,"n_draws":1000,"fdr_within_panel":0.7372627373,"fdr_global":0.5266162409},{"panel":"NO_EDGE_TYPE_CALIBRATION","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.3522727273,"null_mean":0.1743901515,"null_sd":0.0228832949,"fold_over_null":2.0200264993,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_EDGE_TYPE_CALIBRATION","term":"Basal-like","statistic":"both_endpoints","observed":0.0606060606,"null_mean":0.0106363636,"null_sd":0.0065192223,"fold_over_null":5.698005698,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_EDGE_TYPE_CALIBRATION","term":"Basal-like","statistic":"unique_node_fraction","observed":0.2083333333,"null_mean":0.0933462014,"null_sd":0.0119038613,"fold_over_null":2.2318351503,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_SIGNED_WEIGHT","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.4227941176,"null_mean":0.1801360294,"null_sd":0.0231613795,"fold_over_null":2.3470824744,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_SIGNED_WEIGHT","term":"Basal-like","statistic":"both_endpoints","observed":0.0992647059,"null_mean":0.0114705882,"null_sd":0.0063896812,"fold_over_null":8.6538461538,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512},{"panel":"NO_SIGNED_WEIGHT","term":"Basal-like","statistic":"unique_node_fraction","observed":0.2663755459,"null_mean":0.095443435,"null_sd":0.0120507242,"fold_over_null":2.7909258059,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.000999001,"fdr_global":0.0012487512}],"consensus_tests":[{"panel":"LumA","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.2722117202,"null_mean":0.2056181474,"null_sd":0.0175123018,"fold_over_null":1.3238701136,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.005994006,"fdr_global":0.0095904096},{"panel":"LumA","term":"Basal-like","statistic":"both_endpoints","observed":0.0264650284,"null_mean":0.0138582231,"null_sd":0.005057761,"fold_over_null":1.9096985404,"empirical_p":0.013986014,"n_draws":1000,"fdr_within_panel":0.0559440559,"fdr_global":0.0959040959},{"panel":"LumA","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1141199226,"null_mean":0.1006917312,"null_sd":0.0089859365,"fold_over_null":1.1333594252,"empirical_p":0.0679320679,"n_draws":1000,"fdr_within_panel":0.163036963,"fdr_global":0.2964308419},{"panel":"LumA","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.1417769376,"null_mean":0.0616710775,"null_sd":0.0103481567,"fold_over_null":2.2989210397,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.005994006,"fdr_global":0.0095904096},{"panel":"LumA","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0018903592,"null_mean":0.0012457467,"null_sd":0.001530344,"fold_over_null":1.5174506829,"empirical_p":0.4815184815,"n_draws":1000,"fdr_within_panel":0.642024642,"fdr_global":1.0},{"panel":"LumA","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0348162476,"null_mean":0.0292362889,"null_sd":0.0049979443,"fold_over_null":1.1908572835,"empirical_p":0.1378621379,"n_draws":1000,"fdr_within_panel":0.2067932068,"fdr_global":0.4411588412},{"panel":"LumA","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0189035917,"null_mean":0.0109413989,"null_sd":0.0045709584,"fold_over_null":1.7277125086,"empirical_p":0.0659340659,"n_draws":1000,"fdr_within_panel":0.163036963,"fdr_global":0.2964308419},{"panel":"LumA","term":"LumA","statistic":"both_endpoints","observed":0.0018903592,"null_mean":0.0002230624,"null_sd":0.0006387909,"fold_over_null":8.4745762712,"empirical_p":0.1138861139,"n_draws":1000,"fdr_within_panel":0.1952333381,"fdr_global":0.4205025743},{"panel":"LumA","term":"LumA","statistic":"unique_node_fraction","observed":0.0096711799,"null_mean":0.0061510398,"null_sd":0.00243059,"fold_over_null":1.5722837414,"empirical_p":0.0849150849,"n_draws":1000,"fdr_within_panel":0.1698301698,"fdr_global":0.3396603397},{"panel":"LumA","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0056710775,"null_mean":0.0190491493,"null_sd":0.0058235832,"fold_over_null":0.2977076511,"empirical_p":0.997002997,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumA","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":7.18336e-05,"null_sd":0.0003901593,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumA","term":"LumB","statistic":"unique_node_fraction","observed":0.0058027079,"null_mean":0.0107550445,"null_sd":0.0031664741,"fold_over_null":0.5395336027,"empirical_p":0.958041958,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.2381852552,"null_mean":0.1940359168,"null_sd":0.0163714249,"fold_over_null":1.2275317843,"empirical_p":0.005994006,"n_draws":1000,"fdr_within_panel":0.0719280719,"fdr_global":0.047952048},{"panel":"LumB","term":"Basal-like","statistic":"both_endpoints","observed":0.011342155,"null_mean":0.0124763705,"null_sd":0.0046234576,"fold_over_null":0.9090909091,"empirical_p":0.6633366633,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1071428571,"null_mean":0.097177183,"null_sd":0.0085046219,"fold_over_null":1.1025515851,"empirical_p":0.1278721279,"n_draws":1000,"fdr_within_panel":0.7672327672,"fdr_global":0.4384187241},{"panel":"LumB","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0491493384,"null_mean":0.0605274102,"null_sd":0.0099889987,"fold_over_null":0.8120178644,"empirical_p":0.8921078921,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0014480151,"null_sd":0.0015950375,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0218253968,"null_mean":0.0303718383,"null_sd":0.0049404118,"fold_over_null":0.7186063812,"empirical_p":0.956043956,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0075614367,"null_mean":0.0108223062,"null_sd":0.0042288223,"fold_over_null":0.6986899563,"empirical_p":0.8321678322,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":0.000100189,"null_sd":0.0004237157,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"LumA","statistic":"unique_node_fraction","observed":0.005952381,"null_mean":0.0062449553,"null_sd":0.0023169001,"fold_over_null":0.9531502938,"empirical_p":0.5314685315,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0056710775,"null_mean":0.0185746692,"null_sd":0.0057365262,"fold_over_null":0.3053124364,"empirical_p":0.998001998,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001833648,"null_sd":0.0005908353,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"LumB","term":"LumB","statistic":"unique_node_fraction","observed":0.005952381,"null_mean":0.011017139,"null_sd":0.0033024271,"fold_over_null":0.5402837316,"empirical_p":0.9440559441,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.1342155009,"null_mean":0.1895652174,"null_sd":0.016346785,"fold_over_null":0.7080175509,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"Basal-like","statistic":"both_endpoints","observed":0.0056710775,"null_mean":0.0107013233,"null_sd":0.0044153281,"fold_over_null":0.5299417064,"empirical_p":0.9180819181,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"Basal-like","statistic":"unique_node_fraction","observed":0.0652985075,"null_mean":0.0972553729,"null_sd":0.0084477501,"fold_over_null":0.6714128541,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0529300567,"null_mean":0.0579773157,"null_sd":0.0100654217,"fold_over_null":0.9129442452,"empirical_p":0.7232767233,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0014763705,"null_sd":0.0016324921,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0298507463,"null_mean":0.0294962323,"null_sd":0.0047737711,"fold_over_null":1.0120189576,"empirical_p":0.4625374625,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"LumA","statistic":"at_least_one_endpoint","observed":0.011342155,"null_mean":0.0114347826,"null_sd":0.0045692141,"fold_over_null":0.9918994875,"empirical_p":0.5724275724,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":0.000100189,"null_sd":0.0004320753,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"LumA","statistic":"unique_node_fraction","observed":0.0074626866,"null_mean":0.006036169,"null_sd":0.0022450387,"fold_over_null":1.2363283034,"empirical_p":0.3036963037,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":0.8098568099},{"panel":"HER2-enriched","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0170132325,"null_mean":0.0181190926,"null_sd":0.0057405938,"fold_over_null":0.9389671362,"empirical_p":0.6163836164,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001266541,"null_sd":0.0004728679,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"HER2-enriched","term":"LumB","statistic":"unique_node_fraction","observed":0.0130597015,"null_mean":0.0102106862,"null_sd":0.0031201467,"fold_over_null":1.2790228972,"empirical_p":0.1688311688,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":0.4766997708},{"panel":"Basal-like","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.4215500945,"null_mean":0.1885198488,"null_sd":0.0163002048,"fold_over_null":2.2361045655,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.0095904096},{"panel":"Basal-like","term":"Basal-like","statistic":"both_endpoints","observed":0.0718336484,"null_mean":0.0125444234,"null_sd":0.0046564161,"fold_over_null":5.7263411694,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.0095904096},{"panel":"Basal-like","term":"Basal-like","statistic":"unique_node_fraction","observed":0.2291666667,"null_mean":0.0965231938,"null_sd":0.0081999424,"fold_over_null":2.3742134668,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.0095904096},{"panel":"Basal-like","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0604914934,"null_mean":0.0642060491,"null_sd":0.0105973478,"fold_over_null":0.9421463271,"empirical_p":0.6593406593,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"Basal-like","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0014971645,"null_sd":0.0016881802,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"Basal-like","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0208333333,"null_mean":0.0305313728,"null_sd":0.0047541752,"fold_over_null":0.6823582236,"empirical_p":0.984015984,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"Basal-like","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0056710775,"null_mean":0.01036862,"null_sd":0.004451513,"fold_over_null":0.546946217,"empirical_p":0.9100899101,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"Basal-like","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":6.42722e-05,"null_sd":0.0003427596,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"Basal-like","term":"LumA","statistic":"unique_node_fraction","observed":0.0041666667,"null_mean":0.0058561019,"null_sd":0.002378303,"fold_over_null":0.7115085674,"empirical_p":0.7432567433,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"Basal-like","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0321361059,"null_mean":0.0197807183,"null_sd":0.005964273,"fold_over_null":1.624617737,"empirical_p":0.031968032,"n_draws":1000,"fdr_within_panel":0.0959040959,"fdr_global":0.1918081918},{"panel":"Basal-like","term":"LumB","statistic":"both_endpoints","observed":0.0018903592,"null_mean":0.0003308129,"null_sd":0.0007852333,"fold_over_null":5.7142857143,"empirical_p":0.1618381618,"n_draws":1000,"fdr_within_panel":0.3236763237,"fdr_global":0.4766997708},{"panel":"Basal-like","term":"LumB","statistic":"unique_node_fraction","observed":0.0166666667,"null_mean":0.0109682983,"null_sd":0.0032187878,"fold_over_null":1.5195307633,"empirical_p":0.042957043,"n_draws":1000,"fdr_within_panel":0.1030969031,"fdr_global":0.2291042291}],"q_tests":[{"panel":"q_raw::LumA","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.247637051,"null_mean":0.1909338374,"null_sd":0.0167433365,"fold_over_null":1.2969783375,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.005047584},{"panel":"q_raw::LumA","term":"Basal-like","statistic":"both_endpoints","observed":0.0170132325,"null_mean":0.0132533081,"null_sd":0.004714959,"fold_over_null":1.2836970475,"empirical_p":0.2797202797,"n_draws":1000,"fdr_within_panel":0.372960373,"fdr_global":0.4662923284},{"panel":"q_raw::LumA","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1198428291,"null_mean":0.0980352099,"null_sd":0.0083048595,"fold_over_null":1.2224468041,"empirical_p":0.004995005,"n_draws":1000,"fdr_within_panel":0.014985015,"fdr_global":0.0184430954},{"panel":"q_raw::LumA","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.1285444234,"null_mean":0.0598393195,"null_sd":0.0097870194,"fold_over_null":2.1481598484,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.005047584},{"panel":"q_raw::LumA","term":"HER2-enriched","statistic":"both_endpoints","observed":0.011342155,"null_mean":0.0011323251,"null_sd":0.0014652415,"fold_over_null":10.0166944908,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.003996004,"fdr_global":0.005047584},{"panel":"q_raw::LumA","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0333988212,"null_mean":0.0283943663,"null_sd":0.0044752974,"fold_over_null":1.1762481625,"empirical_p":0.1328671329,"n_draws":1000,"fdr_within_panel":0.1993006993,"fdr_global":0.2834498834},{"panel":"q_raw::LumA","term":"LumA","statistic":"at_least_one_endpoint","observed":0.02268431,"null_mean":0.0121493384,"null_sd":0.0045172787,"fold_over_null":1.8671230745,"empirical_p":0.02997003,"n_draws":1000,"fdr_within_panel":0.0513771942,"fdr_global":0.0899100899},{"panel":"q_raw::LumA","term":"LumA","statistic":"both_endpoints","observed":0.0037807183,"null_mean":0.0002514178,"null_sd":0.0006957098,"fold_over_null":15.037593985,"empirical_p":0.010989011,"n_draws":1000,"fdr_within_panel":0.0263736264,"fdr_global":0.0376766091},{"panel":"q_raw::LumA","term":"LumA","statistic":"unique_node_fraction","observed":0.0117878193,"null_mean":0.0063997377,"null_sd":0.0022133029,"fold_over_null":1.8419222463,"empirical_p":0.020979021,"n_draws":1000,"fdr_within_panel":0.041958042,"fdr_global":0.0649672908},{"panel":"q_raw::LumA","term":"LumB","statistic":"at_least_one_endpoint","observed":0.011342155,"null_mean":0.0208241966,"null_sd":0.0060426365,"fold_over_null":0.5446623094,"empirical_p":0.966033966,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::LumA","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001323251,"null_sd":0.0004971651,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::LumA","term":"LumB","statistic":"unique_node_fraction","observed":0.0078585462,"null_mean":0.0107699911,"null_sd":0.0030042,"fold_over_null":0.7296706272,"empirical_p":0.8351648352,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::LumA","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.2911153119,"null_mean":0.1978449905,"null_sd":0.016537016,"fold_over_null":1.4714313014,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::LumA","term":"Basal-like","statistic":"both_endpoints","observed":0.0245746692,"null_mean":0.0134744802,"null_sd":0.0050166367,"fold_over_null":1.8237934905,"empirical_p":0.030969031,"n_draws":1000,"fdr_within_panel":0.0464535465,"fdr_global":0.0900917265},{"panel":"q_module::LumA","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1401869159,"null_mean":0.10016551,"null_sd":0.0084303222,"fold_over_null":1.3995527594,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::LumA","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.1493383743,"null_mean":0.0598979206,"null_sd":0.0098281288,"fold_over_null":2.4932146689,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::LumA","term":"HER2-enriched","statistic":"both_endpoints","observed":0.011342155,"null_mean":0.0010075614,"null_sd":0.0013385724,"fold_over_null":11.2570356473,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::LumA","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0280373832,"null_mean":0.0289125499,"null_sd":0.0046565082,"fold_over_null":0.9697305598,"empirical_p":0.5684315684,"n_draws":1000,"fdr_within_panel":0.7579087579,"fdr_global":0.8526473526},{"panel":"q_module::LumA","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0283553875,"null_mean":0.0116597353,"null_sd":0.0044731568,"fold_over_null":2.4319066148,"empirical_p":0.002997003,"n_draws":1000,"fdr_within_panel":0.005994006,"fdr_global":0.0130778313},{"panel":"q_module::LumA","term":"LumA","statistic":"both_endpoints","observed":0.0037807183,"null_mean":0.0002381853,"null_sd":0.0006609401,"fold_over_null":15.873015873,"empirical_p":0.006993007,"n_draws":1000,"fdr_within_panel":0.011988012,"fdr_global":0.0248640249},{"panel":"q_module::LumA","term":"LumA","statistic":"unique_node_fraction","observed":0.0163551402,"null_mean":0.0064171405,"null_sd":0.0023106571,"fold_over_null":2.5486647986,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::LumA","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0094517958,"null_mean":0.0195028355,"null_sd":0.0058839078,"fold_over_null":0.4846370069,"empirical_p":0.981018981,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::LumA","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001323251,"null_sd":0.0004899174,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::LumA","term":"LumB","statistic":"unique_node_fraction","observed":0.0070093458,"null_mean":0.0107408918,"null_sd":0.0030913938,"fold_over_null":0.6525850847,"empirical_p":0.8811188811,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::LumB","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.190926276,"null_mean":0.1773913043,"null_sd":0.015708356,"fold_over_null":1.0763000853,"empirical_p":0.2157842158,"n_draws":1000,"fdr_within_panel":0.5442557443,"fdr_global":0.3983708599},{"panel":"q_raw::LumB","term":"Basal-like","statistic":"both_endpoints","observed":0.0037807183,"null_mean":0.0108544423,"null_sd":0.0045175816,"fold_over_null":0.3483106931,"empirical_p":0.98001998,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::LumB","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1011673152,"null_mean":0.0949982999,"null_sd":0.0084545156,"fold_over_null":1.0649381652,"empirical_p":0.2267732268,"n_draws":1000,"fdr_within_panel":0.5442557443,"fdr_global":0.4107590523},{"panel":"q_raw::LumB","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0812854442,"null_mean":0.0627334594,"null_sd":0.0099805377,"fold_over_null":1.2957271138,"empirical_p":0.03996004,"n_draws":1000,"fdr_within_panel":0.1838161838,"fdr_global":0.1036801037},{"panel":"q_raw::LumB","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0018903592,"null_mean":0.0017561437,"null_sd":0.0017761751,"fold_over_null":1.0764262648,"empirical_p":0.6133866134,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":0.8921987104},{"panel":"q_raw::LumB","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0389105058,"null_mean":0.0309519836,"null_sd":0.0045744389,"fold_over_null":1.2571247882,"empirical_p":0.045954046,"n_draws":1000,"fdr_within_panel":0.1838161838,"fdr_global":0.1131176516},{"panel":"q_raw::LumB","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0037807183,"null_mean":0.0102117202,"null_sd":0.0041884598,"fold_over_null":0.3702332469,"empirical_p":0.977022977,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::LumB","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":4.53686e-05,"null_sd":0.0002894624,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::LumB","term":"LumA","statistic":"unique_node_fraction","observed":0.0038910506,"null_mean":0.0056127433,"null_sd":0.0021845209,"fold_over_null":0.6932529045,"empirical_p":0.7082917083,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":0.9999412352},{"panel":"q_raw::LumB","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0302457467,"null_mean":0.0193667297,"null_sd":0.0057294292,"fold_over_null":1.5617374329,"empirical_p":0.043956044,"n_draws":1000,"fdr_within_panel":0.1838161838,"fdr_global":0.1110468479},{"panel":"q_raw::LumB","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":4.15879e-05,"null_sd":0.0002774229,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::LumB","term":"LumB","statistic":"unique_node_fraction","observed":0.0116731518,"null_mean":0.0101557488,"null_sd":0.0027828852,"fold_over_null":1.1494132064,"empirical_p":0.2797202797,"n_draws":1000,"fdr_within_panel":0.5594405594,"fdr_global":0.4662923284},{"panel":"q_module::LumB","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.1984877127,"null_mean":0.1778827977,"null_sd":0.0158341885,"fold_over_null":1.1158342189,"empirical_p":0.1058941059,"n_draws":1000,"fdr_within_panel":0.4035964036,"fdr_global":0.2420436706},{"panel":"q_module::LumB","term":"Basal-like","statistic":"both_endpoints","observed":0.0075614367,"null_mean":0.0112797732,"null_sd":0.0046048175,"fold_over_null":0.6703536115,"empirical_p":0.8601398601,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::LumB","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1015801354,"null_mean":0.0949869497,"null_sd":0.0081577781,"fold_over_null":1.0694114902,"empirical_p":0.2017982018,"n_draws":1000,"fdr_within_panel":0.4035964036,"fdr_global":0.3798554387},{"panel":"q_module::LumB","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0888468809,"null_mean":0.0622268431,"null_sd":0.0105989512,"fold_over_null":1.4277902667,"empirical_p":0.014985015,"n_draws":1000,"fdr_within_panel":0.1798201798,"fdr_global":0.047952048},{"panel":"q_module::LumB","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0015746692,"null_sd":0.0017550567,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::LumB","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0361173815,"null_mean":0.0309782825,"null_sd":0.0047730608,"fold_over_null":1.1658936042,"empirical_p":0.1458541459,"n_draws":1000,"fdr_within_panel":0.4035964036,"fdr_global":0.3043912609},{"panel":"q_module::LumB","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0037807183,"null_mean":0.0099470699,"null_sd":0.0042942855,"fold_over_null":0.3800836184,"empirical_p":0.974025974,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::LumB","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":2.45747e-05,"null_sd":0.0002142357,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::LumB","term":"LumA","statistic":"unique_node_fraction","observed":0.0045146727,"null_mean":0.0055892803,"null_sd":0.0022389526,"fold_over_null":0.8077377432,"empirical_p":0.6783216783,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":0.9719235988},{"panel":"q_module::LumB","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0321361059,"null_mean":0.0198204159,"null_sd":0.0058800352,"fold_over_null":1.6213638531,"empirical_p":0.034965035,"n_draws":1000,"fdr_within_panel":0.2097902098,"fdr_global":0.0959040959},{"panel":"q_module::LumB","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":6.42722e-05,"null_sd":0.0003427596,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::LumB","term":"LumB","statistic":"unique_node_fraction","observed":0.0135440181,"null_mean":0.0106788523,"null_sd":0.0029677515,"fold_over_null":1.2683027752,"empirical_p":0.1808191808,"n_draws":1000,"fdr_within_panel":0.4035964036,"fdr_global":0.3616383616},{"panel":"q_raw::HER2-enriched","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.1984877127,"null_mean":0.1833780718,"null_sd":0.0164731423,"fold_over_null":1.0823961157,"empirical_p":0.1808191808,"n_draws":1000,"fdr_within_panel":0.3236763237,"fdr_global":0.3616383616},{"panel":"q_raw::HER2-enriched","term":"Basal-like","statistic":"both_endpoints","observed":0.0075614367,"null_mean":0.0104688091,"null_sd":0.0043246649,"fold_over_null":0.7222824124,"empirical_p":0.8241758242,"n_draws":1000,"fdr_within_panel":0.989010989,"fdr_global":1.0},{"panel":"q_raw::HER2-enriched","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1128107075,"null_mean":0.0931187185,"null_sd":0.0084785943,"fold_over_null":1.2114718637,"empirical_p":0.012987013,"n_draws":1000,"fdr_within_panel":0.038961039,"fdr_global":0.0429914913},{"panel":"q_raw::HER2-enriched","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0907372401,"null_mean":0.0593345936,"null_sd":0.010118651,"fold_over_null":1.5292468459,"empirical_p":0.004995005,"n_draws":1000,"fdr_within_panel":0.01998002,"fdr_global":0.0184430954},{"panel":"q_raw::HER2-enriched","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0075614367,"null_mean":0.0012570888,"null_sd":0.0014586058,"fold_over_null":6.015037594,"empirical_p":0.003996004,"n_draws":1000,"fdr_within_panel":0.01998002,"fdr_global":0.0166789732},{"panel":"q_raw::HER2-enriched","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0458891013,"null_mean":0.029138971,"null_sd":0.0046291882,"fold_over_null":1.5748360291,"empirical_p":0.002997003,"n_draws":1000,"fdr_within_panel":0.01998002,"fdr_global":0.0130778313},{"panel":"q_raw::HER2-enriched","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0151228733,"null_mean":0.0114820416,"null_sd":0.0045227602,"fold_over_null":1.3170892328,"empirical_p":0.2697302697,"n_draws":1000,"fdr_within_panel":0.3756243756,"fdr_global":0.4662923284},{"panel":"q_raw::HER2-enriched","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001304348,"null_sd":0.0004793591,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::HER2-enriched","term":"LumA","statistic":"unique_node_fraction","observed":0.0076481836,"null_mean":0.0059380153,"null_sd":0.0021486725,"fold_over_null":1.2880033481,"empirical_p":0.1888111888,"n_draws":1000,"fdr_within_panel":0.3236763237,"fdr_global":0.3699157985},{"panel":"q_raw::HER2-enriched","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0245746692,"null_mean":0.0203119093,"null_sd":0.0059055906,"fold_over_null":1.2098650535,"empirical_p":0.2817182817,"n_draws":1000,"fdr_within_panel":0.3756243756,"fdr_global":0.4662923284},{"panel":"q_raw::HER2-enriched","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001077505,"null_sd":0.0004623109,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::HER2-enriched","term":"LumB","statistic":"unique_node_fraction","observed":0.0152963671,"null_mean":0.0107269637,"null_sd":0.0030203615,"fold_over_null":1.4259736038,"empirical_p":0.0759240759,"n_draws":1000,"fdr_within_panel":0.1822177822,"fdr_global":0.1822177822},{"panel":"q_module::HER2-enriched","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.1871455577,"null_mean":0.1889943289,"null_sd":0.017114954,"fold_over_null":0.9902178479,"empirical_p":0.5594405594,"n_draws":1000,"fdr_within_panel":0.9050949051,"fdr_global":0.8526473526},{"panel":"q_module::HER2-enriched","term":"Basal-like","statistic":"both_endpoints","observed":0.0132325142,"null_mean":0.0115557656,"null_sd":0.0045948325,"fold_over_null":1.1451006053,"empirical_p":0.4275724276,"n_draws":1000,"fdr_within_panel":0.8551448551,"fdr_global":0.6729008696},{"panel":"q_module::HER2-enriched","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1101123596,"null_mean":0.094675643,"null_sd":0.0085115437,"fold_over_null":1.1630484464,"empirical_p":0.037962038,"n_draws":1000,"fdr_within_panel":0.1138861139,"fdr_global":0.1012321012},{"panel":"q_module::HER2-enriched","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.1001890359,"null_mean":0.0598960302,"null_sd":0.01021721,"fold_over_null":1.6727157961,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.005994006,"fdr_global":0.005047584},{"panel":"q_module::HER2-enriched","term":"HER2-enriched","statistic":"both_endpoints","observed":0.011342155,"null_mean":0.0014196597,"null_sd":0.0015857654,"fold_over_null":7.9893475366,"empirical_p":0.001998002,"n_draws":1000,"fdr_within_panel":0.007992008,"fdr_global":0.0095904096},{"panel":"q_module::HER2-enriched","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0494382022,"null_mean":0.0298020256,"null_sd":0.0046804803,"fold_over_null":1.6588873161,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.005994006,"fdr_global":0.005047584},{"panel":"q_module::HER2-enriched","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0075614367,"null_mean":0.0109716446,"null_sd":0.0043018705,"fold_over_null":0.6891798759,"empirical_p":0.8431568432,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::HER2-enriched","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001304348,"null_sd":0.000494058,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::HER2-enriched","term":"LumA","statistic":"unique_node_fraction","observed":0.006741573,"null_mean":0.0057321924,"null_sd":0.0021066215,"fold_over_null":1.176089808,"empirical_p":0.2667332667,"n_draws":1000,"fdr_within_panel":0.6401598402,"fdr_global":0.4662923284},{"panel":"q_module::HER2-enriched","term":"LumB","statistic":"at_least_one_endpoint","observed":0.011342155,"null_mean":0.018952741,"null_sd":0.0058033561,"fold_over_null":0.5984440455,"empirical_p":0.9400599401,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::HER2-enriched","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001039698,"null_sd":0.0004393978,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::HER2-enriched","term":"LumB","statistic":"unique_node_fraction","observed":0.008988764,"null_mean":0.0102448747,"null_sd":0.0030081964,"fold_over_null":0.8773913102,"empirical_p":0.6033966034,"n_draws":1000,"fdr_within_panel":0.9050949051,"fdr_global":0.8911703681},{"panel":"q_raw::Basal-like","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.3175803403,"null_mean":0.1798620038,"null_sd":0.0157339521,"fold_over_null":1.7656888814,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.002997003,"fdr_global":0.005047584},{"panel":"q_raw::Basal-like","term":"Basal-like","statistic":"both_endpoints","observed":0.0378071834,"null_mean":0.01068431,"null_sd":0.0045669813,"fold_over_null":3.5385704176,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.002997003,"fdr_global":0.005047584},{"panel":"q_raw::Basal-like","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1959910913,"null_mean":0.0946150505,"null_sd":0.0080725912,"fold_over_null":2.0714578742,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.002997003,"fdr_global":0.005047584},{"panel":"q_raw::Basal-like","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0793950851,"null_mean":0.0639187146,"null_sd":0.0109526047,"fold_over_null":1.2421258096,"empirical_p":0.1048951049,"n_draws":1000,"fdr_within_panel":0.1798201798,"fdr_global":0.2420436706},{"panel":"q_raw::Basal-like","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0017637051,"null_sd":0.0018027134,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::Basal-like","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0311804009,"null_mean":0.0301075993,"null_sd":0.0045781096,"fold_over_null":1.0356322525,"empirical_p":0.4025974026,"n_draws":1000,"fdr_within_panel":0.5367965368,"fdr_global":0.6441558442},{"panel":"q_raw::Basal-like","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0132325142,"null_mean":0.0106030246,"null_sd":0.0043310964,"fold_over_null":1.2479942949,"empirical_p":0.3186813187,"n_draws":1000,"fdr_within_panel":0.478021978,"fdr_global":0.5185323151},{"panel":"q_raw::Basal-like","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":7.75047e-05,"null_sd":0.000384447,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::Basal-like","term":"LumA","statistic":"unique_node_fraction","observed":0.0111358575,"null_mean":0.0060326964,"null_sd":0.002313729,"fold_over_null":1.8459171014,"empirical_p":0.034965035,"n_draws":1000,"fdr_within_panel":0.0699300699,"fdr_global":0.0959040959},{"panel":"q_raw::Basal-like","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0472589792,"null_mean":0.0216351607,"null_sd":0.0062272422,"fold_over_null":2.1843599825,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.002997003,"fdr_global":0.005047584},{"panel":"q_raw::Basal-like","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.0001795841,"null_sd":0.0005673127,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_raw::Basal-like","term":"LumB","statistic":"unique_node_fraction","observed":0.0200445434,"null_mean":0.011000199,"null_sd":0.0030860042,"fold_over_null":1.822198251,"empirical_p":0.004995005,"n_draws":1000,"fdr_within_panel":0.011988012,"fdr_global":0.0184430954},{"panel":"q_module::Basal-like","term":"Basal-like","statistic":"at_least_one_endpoint","observed":0.3213610586,"null_mean":0.1790869565,"null_sd":0.0163732662,"fold_over_null":1.7944414537,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::Basal-like","term":"Basal-like","statistic":"both_endpoints","observed":0.0396975425,"null_mean":0.010436673,"null_sd":0.004329213,"fold_over_null":3.8036587575,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::Basal-like","term":"Basal-like","statistic":"unique_node_fraction","observed":0.1733333333,"null_mean":0.0936089138,"null_sd":0.0084834126,"fold_over_null":1.8516755113,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::Basal-like","term":"HER2-enriched","statistic":"at_least_one_endpoint","observed":0.0793950851,"null_mean":0.0649243856,"null_sd":0.0108897093,"fold_over_null":1.2228854273,"empirical_p":0.1098901099,"n_draws":1000,"fdr_within_panel":0.2197802198,"fdr_global":0.2453360593},{"panel":"q_module::Basal-like","term":"HER2-enriched","statistic":"both_endpoints","observed":0.0,"null_mean":0.0017391304,"null_sd":0.0018176219,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::Basal-like","term":"HER2-enriched","statistic":"unique_node_fraction","observed":0.0293333333,"null_mean":0.0303237164,"null_sd":0.0045549874,"fold_over_null":0.9673396533,"empirical_p":0.5684315684,"n_draws":1000,"fdr_within_panel":0.7579087579,"fdr_global":0.8526473526},{"panel":"q_module::Basal-like","term":"LumA","statistic":"at_least_one_endpoint","observed":0.0151228733,"null_mean":0.0107069943,"null_sd":0.0044181728,"fold_over_null":1.4124293785,"empirical_p":0.1968031968,"n_draws":1000,"fdr_within_panel":0.2952047952,"fdr_global":0.3778621379},{"panel":"q_module::Basal-like","term":"LumA","statistic":"both_endpoints","observed":0.0,"null_mean":2.83554e-05,"null_sd":0.0002298928,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::Basal-like","term":"LumA","statistic":"unique_node_fraction","observed":0.008,"null_mean":0.0059541144,"null_sd":0.002274904,"fold_over_null":1.3436087238,"empirical_p":0.1288711289,"n_draws":1000,"fdr_within_panel":0.2209219352,"fdr_global":0.2811733721},{"panel":"q_module::Basal-like","term":"LumB","statistic":"at_least_one_endpoint","observed":0.0548204159,"null_mean":0.0202173913,"null_sd":0.0057701777,"fold_over_null":2.7115474521,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584},{"panel":"q_module::Basal-like","term":"LumB","statistic":"both_endpoints","observed":0.0,"null_mean":0.000194707,"null_sd":0.0005871916,"fold_over_null":0.0,"empirical_p":1.0,"n_draws":1000,"fdr_within_panel":1.0,"fdr_global":1.0},{"panel":"q_module::Basal-like","term":"LumB","statistic":"unique_node_fraction","observed":0.024,"null_mean":0.0107568348,"null_sd":0.0030957918,"fold_over_null":2.2311396012,"empirical_p":0.000999001,"n_draws":1000,"fdr_within_panel":0.0023976024,"fdr_global":0.005047584}],"kappa":[{"kappa":0.007,"class_name":"LumA","eligible_edges":52878,"k_edges":370,"n_union_edges":733,"strict_edges":119,"strict_nodes":145,"mean_pairwise_jaccard":0.4223038415,"consensus_n_edges":370,"consensus_n_nodes":418,"consensus_n_components":103,"consensus_giant_nodes":101,"consensus_giant_fraction":0.2416267943,"consensus_dyad_components":57,"consensus_degree_one_nodes":268,"consensus_density":0.0021227038,"consensus_reciprocity":0.1891891892,"consensus_mean_total_degree":1.7703349282,"consensus_degree_gini":0.3461463856,"strict_n_edges":119,"strict_n_nodes":145,"strict_n_components":40,"strict_giant_nodes":50,"strict_giant_fraction":0.3448275862,"strict_dyad_components":30,"strict_degree_one_nodes":108,"strict_density":0.0056992337,"strict_reciprocity":0.1176470588,"strict_mean_total_degree":1.6413793103,"strict_degree_gini":0.341292379,"jaccard_with_primary_consensus":0.6898496241,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.007,"class_name":"LumB","eligible_edges":52878,"k_edges":370,"n_union_edges":656,"strict_edges":110,"strict_nodes":137,"mean_pairwise_jaccard":0.4708544126,"consensus_n_edges":370,"consensus_n_nodes":402,"consensus_n_components":103,"consensus_giant_nodes":25,"consensus_giant_fraction":0.0621890547,"consensus_dyad_components":47,"consensus_degree_one_nodes":236,"consensus_density":0.0022952569,"consensus_reciprocity":0.3837837838,"consensus_mean_total_degree":1.8407960199,"consensus_degree_gini":0.3417843216,"strict_n_edges":110,"strict_n_nodes":137,"strict_n_components":46,"strict_giant_nodes":9,"strict_giant_fraction":0.0656934307,"strict_dyad_components":18,"strict_degree_one_nodes":80,"strict_density":0.0059038214,"strict_reciprocity":0.3454545455,"strict_mean_total_degree":1.6058394161,"strict_degree_gini":0.2611147976,"jaccard_with_primary_consensus":0.6994328922,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.007,"class_name":"HER2-enriched","eligible_edges":52878,"k_edges":370,"n_union_edges":679,"strict_edges":157,"strict_nodes":204,"mean_pairwise_jaccard":0.4980591509,"consensus_n_edges":370,"consensus_n_nodes":437,"consensus_n_components":127,"consensus_giant_nodes":20,"consensus_giant_fraction":0.0457665904,"consensus_dyad_components":61,"consensus_degree_one_nodes":268,"consensus_density":0.001941931,"consensus_reciprocity":0.3243243243,"consensus_mean_total_degree":1.6933638444,"consensus_degree_gini":0.3046570598,"strict_n_edges":157,"strict_n_nodes":204,"strict_n_components":72,"strict_giant_nodes":10,"strict_giant_fraction":0.0490196078,"strict_dyad_components":40,"strict_degree_one_nodes":132,"strict_density":0.0037911716,"strict_reciprocity":0.3184713376,"strict_mean_total_degree":1.5392156863,"strict_degree_gini":0.2621143999,"jaccard_with_primary_consensus":0.6994328922,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.007,"class_name":"Basal-like","eligible_edges":52878,"k_edges":370,"n_union_edges":675,"strict_edges":143,"strict_nodes":181,"mean_pairwise_jaccard":0.4826801404,"consensus_n_edges":370,"consensus_n_nodes":403,"consensus_n_components":99,"consensus_giant_nodes":95,"consensus_giant_fraction":0.2357320099,"consensus_dyad_components":51,"consensus_degree_one_nodes":231,"consensus_density":0.002283866,"consensus_reciprocity":0.3405405405,"consensus_mean_total_degree":1.8362282878,"consensus_degree_gini":0.3334249883,"strict_n_edges":143,"strict_n_nodes":181,"strict_n_components":62,"strict_giant_nodes":8,"strict_giant_fraction":0.044198895,"strict_dyad_components":30,"strict_degree_one_nodes":119,"strict_density":0.0043891958,"strict_reciprocity":0.3356643357,"strict_mean_total_degree":1.5801104972,"strict_degree_gini":0.2823474868,"jaccard_with_primary_consensus":0.6962264151,"basal_at_least_one_endpoint_observed":0.5104895105,"basal_at_least_one_endpoint_fold_over_null":2.5809645029,"basal_at_least_one_endpoint_empirical_p":0.000999001,"basal_both_endpoints_observed":0.1258741259,"basal_both_endpoints_fold_over_null":9.7244732577,"basal_both_endpoints_empirical_p":0.000999001,"basal_unique_node_fraction_observed":0.270718232,"basal_unique_node_fraction_fold_over_null":2.6335693978,"basal_unique_node_fraction_empirical_p":0.000999001},{"kappa":0.01,"class_name":"LumA","eligible_edges":52878,"k_edges":529,"n_union_edges":928,"strict_edges":227,"strict_nodes":221,"mean_pairwise_jaccard":0.5127853883,"consensus_n_edges":529,"consensus_n_nodes":517,"consensus_n_components":96,"consensus_giant_nodes":189,"consensus_giant_fraction":0.3655705996,"consensus_dyad_components":43,"consensus_degree_one_nodes":282,"consensus_density":0.0019829667,"consensus_reciprocity":0.3289224953,"consensus_mean_total_degree":2.0464216634,"consensus_degree_gini":0.3762034129,"strict_n_edges":227,"strict_n_nodes":221,"strict_n_components":47,"strict_giant_nodes":68,"strict_giant_fraction":0.3076923077,"strict_dyad_components":23,"strict_degree_one_nodes":121,"strict_density":0.0046688606,"strict_reciprocity":0.3788546256,"strict_mean_total_degree":2.0542986425,"strict_degree_gini":0.3826419758,"jaccard_with_primary_consensus":1.0,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.01,"class_name":"LumB","eligible_edges":52878,"k_edges":529,"n_union_edges":845,"strict_edges":245,"strict_nodes":218,"mean_pairwise_jaccard":0.5704036192,"consensus_n_edges":529,"consensus_n_nodes":504,"consensus_n_components":102,"consensus_giant_nodes":102,"consensus_giant_fraction":0.2023809524,"consensus_dyad_components":44,"consensus_degree_one_nodes":250,"consensus_density":0.0020866862,"consensus_reciprocity":0.4763705104,"consensus_mean_total_degree":2.0992063492,"consensus_degree_gini":0.3696064752,"strict_n_edges":245,"strict_n_nodes":218,"strict_n_components":48,"strict_giant_nodes":29,"strict_giant_fraction":0.1330275229,"strict_dyad_components":11,"strict_degree_one_nodes":82,"strict_density":0.0051790471,"strict_reciprocity":0.612244898,"strict_mean_total_degree":2.247706422,"strict_degree_gini":0.3363976783,"jaccard_with_primary_consensus":1.0,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.01,"class_name":"HER2-enriched","eligible_edges":52878,"k_edges":529,"n_union_edges":865,"strict_edges":288,"strict_nodes":300,"mean_pairwise_jaccard":0.588507739,"consensus_n_edges":529,"consensus_n_nodes":536,"consensus_n_components":124,"consensus_giant_nodes":38,"consensus_giant_fraction":0.0708955224,"consensus_dyad_components":53,"consensus_degree_one_nodes":277,"consensus_density":0.0018447482,"consensus_reciprocity":0.4423440454,"consensus_mean_total_degree":1.973880597,"consensus_degree_gini":0.3449023785,"strict_n_edges":288,"strict_n_nodes":300,"strict_n_components":87,"strict_giant_nodes":20,"strict_giant_fraction":0.0666666667,"strict_dyad_components":33,"strict_degree_one_nodes":139,"strict_density":0.0032107023,"strict_reciprocity":0.5208333333,"strict_mean_total_degree":1.92,"strict_degree_gini":0.3073842593,"jaccard_with_primary_consensus":1.0,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.01,"class_name":"Basal-like","eligible_edges":52878,"k_edges":529,"n_union_edges":825,"strict_edges":284,"strict_nodes":257,"mean_pairwise_jaccard":0.6039258141,"consensus_n_edges":529,"consensus_n_nodes":480,"consensus_n_components":87,"consensus_giant_nodes":209,"consensus_giant_fraction":0.4354166667,"consensus_dyad_components":42,"consensus_degree_one_nodes":239,"consensus_density":0.0023008003,"consensus_reciprocity":0.4612476371,"consensus_mean_total_degree":2.2041666667,"consensus_degree_gini":0.3862752048,"strict_n_edges":284,"strict_n_nodes":257,"strict_n_components":52,"strict_giant_nodes":84,"strict_giant_fraction":0.326848249,"strict_dyad_components":19,"strict_degree_one_nodes":118,"strict_density":0.0043166342,"strict_reciprocity":0.5563380282,"strict_mean_total_degree":2.2101167315,"strict_degree_gini":0.3674576643,"jaccard_with_primary_consensus":1.0,"basal_at_least_one_endpoint_observed":0.4154929577,"basal_at_least_one_endpoint_fold_over_null":2.2152968122,"basal_at_least_one_endpoint_empirical_p":0.000999001,"basal_both_endpoints_observed":0.0774647887,"basal_both_endpoints_fold_over_null":6.8322981366,"basal_both_endpoints_empirical_p":0.000999001,"basal_unique_node_fraction_observed":0.2412451362,"basal_unique_node_fraction_fold_over_null":2.4848629737,"basal_unique_node_fraction_empirical_p":0.000999001},{"kappa":0.015,"class_name":"LumA","eligible_edges":52878,"k_edges":793,"n_union_edges":1265,"strict_edges":399,"strict_nodes":332,"mean_pairwise_jaccard":0.5828954946,"consensus_n_edges":793,"consensus_n_nodes":693,"consensus_n_components":99,"consensus_giant_nodes":408,"consensus_giant_fraction":0.5887445887,"consensus_dyad_components":46,"consensus_degree_one_nodes":339,"consensus_density":0.0016536129,"consensus_reciprocity":0.4110970996,"consensus_mean_total_degree":2.2886002886,"consensus_degree_gini":0.4022989761,"strict_n_edges":399,"strict_n_nodes":332,"strict_n_components":37,"strict_giant_nodes":183,"strict_giant_fraction":0.5512048193,"strict_dyad_components":16,"strict_degree_one_nodes":148,"strict_density":0.0036308375,"strict_reciprocity":0.4411027569,"strict_mean_total_degree":2.4036144578,"strict_degree_gini":0.4082495395,"jaccard_with_primary_consensus":0.6670870113,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.015,"class_name":"LumB","eligible_edges":52878,"k_edges":793,"n_union_edges":1152,"strict_edges":451,"strict_nodes":352,"mean_pairwise_jaccard":0.6447775973,"consensus_n_edges":793,"consensus_n_nodes":658,"consensus_n_components":100,"consensus_giant_nodes":253,"consensus_giant_fraction":0.3844984802,"consensus_dyad_components":39,"consensus_degree_one_nodes":268,"consensus_density":0.0018343488,"consensus_reciprocity":0.5750315259,"consensus_mean_total_degree":2.4103343465,"consensus_degree_gini":0.3907442401,"strict_n_edges":451,"strict_n_nodes":352,"strict_n_components":55,"strict_giant_nodes":51,"strict_giant_fraction":0.1448863636,"strict_dyad_components":10,"strict_degree_one_nodes":107,"strict_density":0.0036502849,"strict_reciprocity":0.6829268293,"strict_mean_total_degree":2.5625,"strict_degree_gini":0.3651985487,"jaccard_with_primary_consensus":0.6670870113,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.015,"class_name":"HER2-enriched","eligible_edges":52878,"k_edges":793,"n_union_edges":1191,"strict_edges":482,"strict_nodes":431,"mean_pairwise_jaccard":0.6458092535,"consensus_n_edges":793,"consensus_n_nodes":715,"consensus_n_components":135,"consensus_giant_nodes":61,"consensus_giant_fraction":0.0853146853,"consensus_dyad_components":53,"consensus_degree_one_nodes":328,"consensus_density":0.0015533486,"consensus_reciprocity":0.5321563682,"consensus_mean_total_degree":2.2181818182,"consensus_degree_gini":0.3719486062,"strict_n_edges":482,"strict_n_nodes":431,"strict_n_components":94,"strict_giant_nodes":30,"strict_giant_fraction":0.0696055684,"strict_dyad_components":28,"strict_degree_one_nodes":170,"strict_density":0.0026007662,"strict_reciprocity":0.5975103734,"strict_mean_total_degree":2.2366589327,"strict_degree_gini":0.3491012891,"jaccard_with_primary_consensus":0.6649874055,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.015,"class_name":"Basal-like","eligible_edges":52878,"k_edges":793,"n_union_edges":1112,"strict_edges":532,"strict_nodes":386,"mean_pairwise_jaccard":0.6988052711,"consensus_n_edges":793,"consensus_n_nodes":613,"consensus_n_components":78,"consensus_giant_nodes":377,"consensus_giant_fraction":0.6150081566,"consensus_dyad_components":36,"consensus_degree_one_nodes":268,"consensus_density":0.0021137873,"consensus_reciprocity":0.5573770492,"consensus_mean_total_degree":2.5872756933,"consensus_degree_gini":0.4290148917,"strict_n_edges":532,"strict_n_nodes":386,"strict_n_components":47,"strict_giant_nodes":242,"strict_giant_fraction":0.6269430052,"strict_dyad_components":20,"strict_degree_one_nodes":141,"strict_density":0.0035798398,"strict_reciprocity":0.6691729323,"strict_mean_total_degree":2.7564766839,"strict_degree_gini":0.4178629865,"jaccard_with_primary_consensus":0.6670870113,"basal_at_least_one_endpoint_observed":0.3364661654,"basal_at_least_one_endpoint_fold_over_null":1.7991215462,"basal_at_least_one_endpoint_empirical_p":0.000999001,"basal_both_endpoints_observed":0.0488721805,"basal_both_endpoints_fold_over_null":4.3918918919,"basal_both_endpoints_empirical_p":0.000999001,"basal_unique_node_fraction_observed":0.2020725389,"basal_unique_node_fraction_fold_over_null":2.1113562029,"basal_unique_node_fraction_empirical_p":0.000999001},{"kappa":0.02,"class_name":"LumA","eligible_edges":52878,"k_edges":1058,"n_union_edges":1564,"strict_edges":649,"strict_nodes":490,"mean_pairwise_jaccard":0.6531056174,"consensus_n_edges":1058,"consensus_n_nodes":842,"consensus_n_components":94,"consensus_giant_nodes":589,"consensus_giant_fraction":0.6995249406,"consensus_dyad_components":45,"consensus_degree_one_nodes":357,"consensus_density":0.0014940928,"consensus_reciprocity":0.4763705104,"consensus_mean_total_degree":2.513064133,"consensus_degree_gini":0.4108163568,"strict_n_edges":649,"strict_n_nodes":490,"strict_n_components":39,"strict_giant_nodes":371,"strict_giant_fraction":0.7571428571,"strict_dyad_components":16,"strict_degree_one_nodes":189,"strict_density":0.0027085681,"strict_reciprocity":0.5146379045,"strict_mean_total_degree":2.6489795918,"strict_degree_gini":0.4154775007,"jaccard_with_primary_consensus":0.5,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.02,"class_name":"LumB","eligible_edges":52878,"k_edges":1058,"n_union_edges":1402,"strict_edges":690,"strict_nodes":508,"mean_pairwise_jaccard":0.7148624367,"consensus_n_edges":1058,"consensus_n_nodes":814,"consensus_n_components":99,"consensus_giant_nodes":434,"consensus_giant_fraction":0.5331695332,"consensus_dyad_components":38,"consensus_degree_one_nodes":295,"consensus_density":0.0015987138,"consensus_reciprocity":0.618147448,"consensus_mean_total_degree":2.5995085995,"consensus_degree_gini":0.4015248278,"strict_n_edges":690,"strict_n_nodes":508,"strict_n_components":55,"strict_giant_nodes":167,"strict_giant_fraction":0.3287401575,"strict_dyad_components":12,"strict_degree_one_nodes":154,"strict_density":0.002679029,"strict_reciprocity":0.6782608696,"strict_mean_total_degree":2.7165354331,"strict_degree_gini":0.3889079083,"jaccard_with_primary_consensus":0.5,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.02,"class_name":"HER2-enriched","eligible_edges":52878,"k_edges":1058,"n_union_edges":1414,"strict_edges":737,"strict_nodes":614,"mean_pairwise_jaccard":0.7317181809,"consensus_n_edges":1058,"consensus_n_nodes":875,"consensus_n_components":135,"consensus_giant_nodes":297,"consensus_giant_fraction":0.3394285714,"consensus_dyad_components":49,"consensus_degree_one_nodes":344,"consensus_density":0.0013834586,"consensus_reciprocity":0.5897920605,"consensus_mean_total_degree":2.4182857143,"consensus_degree_gini":0.379215771,"strict_n_edges":737,"strict_n_nodes":614,"strict_n_components":114,"strict_giant_nodes":47,"strict_giant_fraction":0.0765472313,"strict_dyad_components":28,"strict_degree_one_nodes":215,"strict_density":0.001958117,"strict_reciprocity":0.6404341927,"strict_mean_total_degree":2.4006514658,"strict_degree_gini":0.3555460777,"jaccard_with_primary_consensus":0.4971698113,"basal_at_least_one_endpoint_observed":null,"basal_at_least_one_endpoint_fold_over_null":null,"basal_at_least_one_endpoint_empirical_p":null,"basal_both_endpoints_observed":null,"basal_both_endpoints_fold_over_null":null,"basal_both_endpoints_empirical_p":null,"basal_unique_node_fraction_observed":null,"basal_unique_node_fraction_fold_over_null":null,"basal_unique_node_fraction_empirical_p":null},{"kappa":0.02,"class_name":"Basal-like","eligible_edges":52878,"k_edges":1058,"n_union_edges":1402,"strict_edges":757,"strict_nodes":515,"mean_pairwise_jaccard":0.7408161544,"consensus_n_edges":1058,"consensus_n_nodes":757,"consensus_n_components":83,"consensus_giant_nodes":503,"consensus_giant_fraction":0.6644649934,"consensus_dyad_components":37,"consensus_degree_one_nodes":302,"consensus_density":0.0018487066,"consensus_reciprocity":0.6011342155,"consensus_mean_total_degree":2.7952443857,"consensus_degree_gini":0.4431056828,"strict_n_edges":757,"strict_n_nodes":515,"strict_n_components":49,"strict_giant_nodes":334,"strict_giant_fraction":0.6485436893,"strict_dyad_components":17,"strict_degree_one_nodes":172,"strict_density":0.0028597333,"strict_reciprocity":0.6763540291,"strict_mean_total_degree":2.9398058252,"strict_degree_gini":0.4323197086,"jaccard_with_primary_consensus":0.5,"basal_at_least_one_endpoint_observed":0.3038309115,"basal_at_least_one_endpoint_fold_over_null":1.67327489,"basal_at_least_one_endpoint_empirical_p":0.000999001,"basal_both_endpoints_observed":0.0396301189,"basal_both_endpoints_fold_over_null":3.6755697133,"basal_both_endpoints_empirical_p":0.000999001,"basal_unique_node_fraction_observed":0.1844660194,"basal_unique_node_fraction_fold_over_null":1.9791123362,"basal_unique_node_fraction_empirical_p":0.000999001}]}""")
summary_structural = pd.DataFrame(FIGURE_DATA['summary_structural'])
components = pd.DataFrame(FIGURE_DATA['components'])
basal_strict = pd.DataFrame(FIGURE_DATA['basal_strict_edges'])
strict_edges = {'Basal-like': basal_strict}
hallmark_raw = pd.DataFrame(FIGURE_DATA['hallmark'])
matched = pd.DataFrame(FIGURE_DATA['matched'])
cross = pd.DataFrame(FIGURE_DATA['cross'])
rewired = pd.DataFrame(FIGURE_DATA['rewired'])
external_raw = pd.DataFrame(FIGURE_DATA['external'])
ablation_raw = pd.DataFrame(FIGURE_DATA['ablation'])
ablation_tests = pd.DataFrame(FIGURE_DATA['ablation_tests'])
consensus_tests = pd.DataFrame(FIGURE_DATA['consensus_tests'])
q_tests = pd.DataFrame(FIGURE_DATA['q_tests'])
kappa_raw = pd.DataFrame(FIGURE_DATA['kappa'])

display(summary_structural)

## Figura 1 — Reproducibilidad y topología

Todo el código de los cuatro paneles está en la siguiente celda.

In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 1 =====================
FIG1_TITLE = 'Reproducibility and topology of class-specific networks'
FIG1_PANEL_TITLES = [
    'A  Five-seed reproducibility',
    'B  Connected structure at a common edge budget',
    'C  Largest components of the Basal-like 5/5 core',
    'D  Basal-like core composition',
]
FIG1_BAR_WIDTH = 0.68
FIG1_COMPARISON_WIDTH = 0.36
FIG1_COMPONENT_COLOR = CLASS_COLORS['Basal-like']
FIG1_COMPOSITION_COLORS = [
    '#8C8C8C', '#55A868', '#C44E52', '#4C78A8', '#F58518'
]

fig1, axes = plt.subplots(
    2, 2,
    figsize=FIGSIZE['structural'],
    constrained_layout=True,
)
x = np.arange(len(CLASSES))
class_palette = [CLASS_COLORS[c] for c in CLASSES]
indexed = summary_structural.set_index('class_name').loc[CLASSES]

# Panel A
ax = axes[0, 0]
values = indexed['mean_pairwise_edge_jaccard']
bars = ax.bar(x, values, color=class_palette, width=FIG1_BAR_WIDTH)
for bar, cls in zip(bars, CLASSES):
    n_edges = int(indexed.loc[cls, 'strict_5of5_directed_edges'])
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.015,
        f'{n_edges} edges',
        ha='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xticks(x, DISPLAY_CLASSES)
ax.set_ylim(0, max(0.69, float(values.max()) + 0.08))
ax.set_ylabel('Mean pairwise edge Jaccard')
ax.set_title(FIG1_PANEL_TITLES[0], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel B
ax = axes[0, 1]
width = FIG1_COMPARISON_WIDTH
ax.bar(
    x - width/2,
    indexed['consensus_giant_fraction'],
    width,
    color=class_palette,
    label='Exact-K consensus',
)
ax.bar(
    x + width/2,
    indexed['strict_core_giant_fraction'],
    width,
    color=class_palette,
    alpha=0.42,
    hatch='//',
    label='Strict 5/5 edge core',
)
ax.set_xticks(x, DISPLAY_CLASSES)
ax.set_ylabel('Fraction of nodes in largest component')
ax.set_title(FIG1_PANEL_TITLES[1], loc='left')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)

# Panel C
ax = axes[1, 0]
top_components = components.head(10).sort_values(
    'component_rank', ascending=False
)
bars = ax.barh(
    [f'C{i}' for i in top_components['component_rank']],
    top_components['n_genes'],
    color=FIG1_COMPONENT_COLOR,
    alpha=0.88,
)
for bar, n_edges in zip(bars, top_components['n_directed_edges']):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height()/2,
        f'{int(n_edges)} edges',
        va='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xlabel('Genes in Basal-like strict component')
ax.set_title(FIG1_PANEL_TITLES[2], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel D
ax = axes[1, 1]
basal_strict = strict_edges['Basal-like']
edge_counts = (
    basal_strict['edge_type'].map(EDGE_TYPES).value_counts()
    .reindex(EDGE_TYPES.values()).fillna(0)
)
direction = pd.Series({
    'Class-strengthened': int(
        (basal_strict['median_differential_logit'] >= 0).sum()
    ),
    'Class-attenuated': int(
        (basal_strict['median_differential_logit'] < 0).sum()
    ),
})
labels = list(edge_counts.index) + list(direction.index)
counts = list(edge_counts.astype(int)) + list(direction.astype(int))
bars = ax.barh(
    labels[::-1], counts[::-1], color=FIG1_COMPOSITION_COLORS[::-1]
)
for bar in bars:
    ax.text(
        bar.get_width() + 3,
        bar.get_y() + bar.get_height()/2,
        f'{int(bar.get_width())}',
        va='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xlim(0, max(counts) * 1.18)
ax.set_xlabel('Strict 5/5 directed edges')
ax.set_title(FIG1_PANEL_TITLES[3], loc='left')
ax.spines[['top', 'right']].set_visible(False)

fig1.suptitle(
    FIG1_TITLE,
    fontsize=STYLE['figure_title_size'],
    fontweight='bold',
)
export_figure(fig1, 'Figure_structural_recovery')
display(fig1)

## Figura 2 — Organización del núcleo Basal-like 5/5

Puedes editar la lista de genes etiquetados, el tamaño de los nodos, los
colores de nodo y arista, la curvatura de las flechas y las leyendas.

In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 2 =====================
FIG2_TITLE = 'Organization of the Basal-like 5/5 edge core'
FIG2_LABELLED_GENES = {
    'EGFR', 'PCNA', 'VGLL1', 'TEAD4', 'KRT5', 'KRT16', 'SOX10',
    'E2F1', 'TFDP1', 'PLK1', 'CDK1', 'NOTCH1', 'CEP76', 'FZD7',
}
FIG2_NODE_COLORS = {
    'signature_and_hallmark': '#C44E52',
    'signature_only': '#F58518',
    'hallmark_only': '#8064A2',
    'other': '#C7C7C7',
}
FIG2_EDGE_COLORS = {
    0: '#9E9E9E',
    1: '#55A868',
    2: '#C44E52',
}
FIG2_NODE_BASE_SIZE = 65
FIG2_NODE_DEGREE_SCALE = 20
FIG2_EDGE_ALPHA = 0.42
FIG2_EDGE_WIDTH = 0.65
FIG2_EDGE_CURVATURE = 0.04

strict = strict_edges['Basal-like']
edge_rows = [
    (row.source_gene, row.target_gene, int(row.edge_type))
    for row in strict.itertuples(index=False)
]
largest_nodes = component_sets(
    [(source, target) for source, target, _ in edge_rows]
)[0]
sub_edges = [
    (source, target, edge_type)
    for source, target, edge_type in edge_rows
    if source in largest_nodes and target in largest_nodes
]
sub_nodes = sorted(largest_nodes)

hallmark = hallmark_raw.copy().sort_values('fold_enrichment', ascending=True)

hallmark_genes = set()
for value in hallmark['overlap_genes'].dropna():
    hallmark_genes.update(str(value).split(';'))
signature_genes = set(
    str(components.iloc[0]['basal_signature_genes']).split(';')
)

fig2 = plt.figure(
    figsize=FIGSIZE['basal_core'],
    constrained_layout=True,
)
grid = fig2.add_gridspec(1, 2, width_ratios=[1.65, 1.0])

# Panel A: red dirigida
ax = fig2.add_subplot(grid[0, 0])
positions = force_layout(sub_nodes, sub_edges)
degree = {node: 0 for node in sub_nodes}
for source, target, _ in sub_edges:
    degree[source] += 1
    degree[target] += 1

node_colors = []
node_sizes = []
for node in sub_nodes:
    in_signature = node in signature_genes
    in_hallmark = node in hallmark_genes
    if in_signature and in_hallmark:
        node_colors.append(FIG2_NODE_COLORS['signature_and_hallmark'])
    elif in_signature:
        node_colors.append(FIG2_NODE_COLORS['signature_only'])
    elif in_hallmark:
        node_colors.append(FIG2_NODE_COLORS['hallmark_only'])
    else:
        node_colors.append(FIG2_NODE_COLORS['other'])
    node_sizes.append(
        FIG2_NODE_BASE_SIZE + FIG2_NODE_DEGREE_SCALE * degree[node]
    )

for edge_type, color in FIG2_EDGE_COLORS.items():
    for source, target, current_type in sub_edges:
        if current_type != edge_type:
            continue
        arrow = FancyArrowPatch(
            positions[source],
            positions[target],
            arrowstyle='-|>',
            mutation_scale=5.5,
            linewidth=FIG2_EDGE_WIDTH,
            color=color,
            alpha=FIG2_EDGE_ALPHA,
            connectionstyle=f'arc3,rad={FIG2_EDGE_CURVATURE}',
            shrinkA=3.2,
            shrinkB=3.2,
        )
        ax.add_patch(arrow)

coordinates = np.vstack([positions[node] for node in sub_nodes])
ax.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=node_colors,
    s=node_sizes,
    linewidths=0.45,
    edgecolors='white',
    zorder=3,
)
for node in sub_nodes:
    if node not in FIG2_LABELLED_GENES:
        continue
    label = ax.text(
        positions[node][0],
        positions[node][1],
        node,
        fontsize=STYLE['gene_label_size'],
        fontweight='bold',
        ha='center',
        va='center',
        zorder=4,
    )
    label.set_path_effects([
        patheffects.withStroke(linewidth=2.5, foreground='white')
    ])

ax.set_title(
    f'A  Largest Basal-like strict component\n'
    f'{len(sub_nodes)} genes, {len(sub_edges)} directed edges',
    loc='left',
)
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=FIG2_NODE_COLORS['signature_and_hallmark'],
           markeredgecolor='white', markersize=7,
           label='Basal signature + Hallmark'),
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=FIG2_NODE_COLORS['signature_only'],
           markeredgecolor='white', markersize=7,
           label='Basal signature only'),
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=FIG2_NODE_COLORS['hallmark_only'],
           markeredgecolor='white', markersize=7,
           label='Hallmark only'),
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=FIG2_NODE_COLORS['other'],
           markeredgecolor='white', markersize=7,
           label='Neither'),
    Line2D([0], [0], color=FIG2_EDGE_COLORS[0], lw=1.2, label='HuRI'),
    Line2D([0], [0], color=FIG2_EDGE_COLORS[1], lw=1.2,
           label='OmniPath activation'),
    Line2D([0], [0], color=FIG2_EDGE_COLORS[2], lw=1.2,
           label='OmniPath inhibition'),
]
ax.legend(
    handles=legend_handles,
    loc='lower right',
    frameon=False,
    fontsize=7.3,
    ncol=2,
    handletextpad=0.5,
    columnspacing=0.9,
)
ax.axis('off')

# Panel B: Hallmarks
right = grid[0, 1].subgridspec(2, 1, height_ratios=[1.18, 1.0])
ax2 = fig2.add_subplot(right[0, 0])
bars = ax2.barh(
    [clean_term(term) for term in hallmark['term']],
    hallmark['fold_enrichment'],
    color=FIG2_NODE_COLORS['hallmark_only'],
)
for bar, overlap, fdr in zip(
    bars, hallmark['overlap_n'], hallmark['fdr']
):
    ax2.text(
        bar.get_width() + 0.08,
        bar.get_y() + bar.get_height()/2,
        f'n={int(overlap)}; FDR={fdr:.3g}',
        va='center',
        fontsize=STYLE['annotation_size'],
    )
ax2.set_xlim(0, float(hallmark['fold_enrichment'].max()) * 1.42)
ax2.set_xlabel('Fold enrichment')
ax2.set_title(
    'B  Significant Hallmarks in the 257-gene 5/5 core',
    loc='left',
)
ax2.spines[['top', 'right']].set_visible(False)

# Panel C: perfil de componentes
ax3 = fig2.add_subplot(right[1, 0])
top_components = components.head(10).sort_values(
    'component_rank', ascending=False
)
bars = ax3.barh(
    [f'C{i}' for i in top_components['component_rank']],
    top_components['n_genes'],
    color=CLASS_COLORS['Basal-like'],
)
for bar, n_signature in zip(
    bars, top_components['basal_signature_genes_n']
):
    ax3.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height()/2,
        f'{int(n_signature)} signature genes',
        va='center',
        fontsize=STYLE['annotation_size'],
    )
ax3.set_xlim(0, float(top_components['n_genes'].max()) * 1.35)
ax3.set_xlabel('Genes')
ax3.set_title('C  Component-size profile', loc='left')
ax3.spines[['top', 'right']].set_visible(False)

fig2.suptitle(
    FIG2_TITLE,
    fontsize=STYLE['figure_title_size'],
    fontweight='bold',
)
export_figure(fig2, 'Figure_Basal_strict_core')
display(fig2)

## Figura 3 — Especificidad y coherencia funcional

La matriz usa la firma direccional corregida y el panel inferior los 1,000
rewirings dirigidos que preservan grado, tipo y signo.

In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 3 =====================
FIG3_TITLE = 'Subtype-specific recovery and functional coherence'
FIG3_PANEL_TITLES = [
    'A  Cross-class strict-core specificity',
    'B  Directional Basal-like recovery',
    'C  Functional coherence beyond preserved topology',
]
FIG3_HEATMAP_MIN = 0.0
FIG3_HEATMAP_MAX = 0.28
FIG3_NULL_COLOR = NEUTRAL
FIG3_REWIRING_ERROR_COLOR = NEUTRAL

fig3 = plt.figure(
    figsize=FIGSIZE['biological_validation'],
    constrained_layout=True,
)
grid = fig3.add_gridspec(2, 2, height_ratios=[1.0, 0.82])

# Panel A: especificidad cruzada
ax = fig3.add_subplot(grid[0, 0])
matrix = (
    cross.pivot(
        index='graph_class',
        columns='annotation_class',
        values='node_fraction',
    ).loc[CLASSES, CLASSES]
)
image = ax.imshow(
    matrix.values,
    cmap=HEATMAP_CMAP,
    vmin=FIG3_HEATMAP_MIN,
    vmax=max(FIG3_HEATMAP_MAX, float(matrix.values.max())),
)
for row in range(4):
    for col in range(4):
        value = matrix.iloc[row, col]
        text_color = 'white' if value < 0.15 else 'black'
        ax.text(
            col, row, f'{value:.3f}',
            ha='center', va='center',
            color=text_color,
            fontsize=STYLE['annotation_size'],
        )
    ax.add_patch(Rectangle(
        (row - 0.48, row - 0.48),
        0.96, 0.96,
        fill=False,
        edgecolor=DIAGONAL_COLOR,
        linewidth=1.8,
    ))
ax.set_xticks(range(4), DISPLAY_CLASSES, rotation=25, ha='right')
ax.set_yticks(range(4), DISPLAY_CLASSES)
ax.set_xlabel('Directional annotation class')
ax.set_ylabel('Strict core class')
ax.set_title(FIG3_PANEL_TITLES[0], loc='left')
fig3.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

# Panel B: firma Basal-like corregida frente al null emparejado
ax = fig3.add_subplot(grid[0, 1])
basal = matched.loc[
    matched['graph_class'].eq('Basal-like')
    & matched['term'].eq('Basal-like')
].copy()
statistic_order = [
    'at_least_one_endpoint',
    'both_endpoints',
    'unique_node_fraction',
]
statistic_labels = ['>=1 endpoint', 'Both endpoints', 'Unique nodes']
basal = basal.set_index('statistic').loc[statistic_order].reset_index()
x = np.arange(3)
width = 0.36
ax.bar(
    x - width/2,
    basal['observed'],
    width,
    color=CLASS_COLORS['Basal-like'],
    label='Observed',
)
ax.bar(
    x + width/2,
    basal['null_mean'],
    width,
    color=FIG3_NULL_COLOR,
    label='Matched-null mean',
)
for index, row in basal.iterrows():
    ax.text(
        index,
        max(row.observed, row.null_mean) + 0.018,
        f'{row.fold_over_null:.2f}x\nFDR={row.fdr_global:.3f}',
        ha='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xticks(x, statistic_labels)
ax.set_ylim(
    0,
    max(
        0.54,
        float(basal[['observed', 'null_mean']].to_numpy().max()) * 1.28,
    ),
)
ax.set_ylabel('Fraction')
ax.set_title(FIG3_PANEL_TITLES[1], loc='left')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)

# Panel C: coherencia funcional frente a 1,000 rewirings
ax = fig3.add_subplot(grid[1, :])
rewiring = rewired.set_index('graph_class').loc[CLASSES]
x = np.arange(4)
ax.bar(
    x - width/2,
    rewiring['observed_coherent_fraction'],
    width,
    color=[CLASS_COLORS[c] for c in CLASSES],
    label='Observed',
)
ax.bar(
    x + width/2,
    rewiring['null_mean'],
    width,
    yerr=rewiring['null_sd'],
    color=FIG3_REWIRING_ERROR_COLOR,
    capsize=3,
    label='Rewired mean +/- SD',
)
for index, row in enumerate(rewiring.itertuples()):
    ax.text(
        index,
        max(row.observed_coherent_fraction, row.null_mean) + 0.009,
        f'{row.fold_over_null:.2f}x',
        ha='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xticks(x, DISPLAY_CLASSES)
ax.set_ylabel('Hallmark-coherent edge fraction')
ax.set_title(FIG3_PANEL_TITLES[2], loc='left')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)

fig3.suptitle(
    FIG3_TITLE,
    fontsize=STYLE['figure_title_size'],
    fontweight='bold',
)
export_figure(fig3, 'Figure_biological_validation')
display(fig3)

## Figura 4 — Concordancia estructural externa

El núcleo de 284 aristas permanece fijo. Se muestran las correlaciones con los
efectos externos y los paneles del null emparejado por grado, tipo y signo.

In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 4 =====================
FIG4_TITLE = 'External concordance of Basal-like edge effects'
FIG4_XLABEL = 'Spearman correlation: development vs external edge effects'
FIG4_SCOPE_ORDER = ['TCGA-BRCA', 'GSE21653', 'GSE1456', 'EXTERNAL_META']
FIG4_SCOPE_LABELS = [
    'TCGA-BRCA',
    'GSE21653',
    'GSE1456',
    'External meta-analysis',
]
FIG4_XLIM = (0.05, 0.76)
FIG4_INTERVAL_COLOR = '#BDBDBD'
FIG4_NULL_COLOR = '#4D4D4D'
FIG4_CORE_COLOR = CLASS_COLORS['Basal-like']
FIG4_LEGEND_ANCHOR = (0.03, 0.56)

external = external_raw.copy()
primary = external.loc[external['is_primary_metric'].astype(bool)].copy()
primary = (
    primary.set_index('scope').loc[FIG4_SCOPE_ORDER].reset_index()
)
y_positions = np.arange(len(primary))[::-1]

fig4, ax = plt.subplots(
    figsize=FIGSIZE['external_concordance'],
    constrained_layout=True,
)
ax.hlines(
    y_positions,
    primary['null_q025'],
    primary['null_q975'],
    color=FIG4_INTERVAL_COLOR,
    lw=7,
    label='Matched-null 95% interval',
)
ax.scatter(
    primary['null_mean'],
    y_positions,
    color=FIG4_NULL_COLOR,
    s=48,
    zorder=3,
    label='Matched-null mean',
)
ax.scatter(
    primary['observed'],
    y_positions,
    color=FIG4_CORE_COLOR,
    s=82,
    zorder=4,
    label='Basal-like 5/5 core',
)
for y, row in zip(y_positions, primary.itertuples(index=False)):
    ax.text(
        min(0.72, row.observed + 0.016),
        y,
        f'FDR={row.fdr_primary_four_scopes:.3f}',
        va='center',
        ha='left',
        fontsize=STYLE['annotation_size'],
    )
ax.set_yticks(y_positions, FIG4_SCOPE_LABELS)
ax.set_xlim(*FIG4_XLIM)
ax.set_xlabel(FIG4_XLABEL)
ax.set_title(FIG4_TITLE, loc='left', fontweight='bold')
ax.legend(
    frameon=False,
    loc='center left',
    bbox_to_anchor=FIG4_LEGEND_ANCHOR,
)
ax.spines[['top', 'right']].set_visible(False)

export_figure(fig4, 'Figure_external_structural_concordance')
display(fig4)

## Figura 5 — Ablaciones, comparador q y sensibilidad a kappa

Los seis paneles están escritos explícitamente: tres ablaciones estructurales,
el comparador de igual presupuesto basado en q y dos paneles de sensibilidad a
la densidad.

In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 5 =====================
FIG5_TITLE = 'Mechanism controls and parameter sensitivity'
FIG5_PANEL_TITLES = [
    'A  Five-seed reproducibility',
    'B  Strict 5/5 core size',
    'C  Directional Basal-like recovery',
    'D  Equal-budget q comparator',
    'E  Density and reproducibility',
    'F  Recovery across edge budgets',
]

ARM_ORDER = [
    'FULL',
    'NO_CONTEXT_LOSS',
    'NO_COHORT_QUERY_LOSS',
    'NO_EDGE_TYPE_CALIBRATION',
    'NO_SIGNED_WEIGHT',
]
ARM_LABELS = [
    'Full', 'No context', 'No cohort query', 'No edge type', 'No signed prior'
]
ARM_COLORS = ['#2166AC', '#B2182B', '#969696', '#969696', '#969696']

Q_COLORS = {
    'InterGATE': CLASS_COLORS['Basal-like'],
    'q raw': '#4C78A8',
    'q + module heuristic': '#72B7B2',
}
KAPPA_LINE_COLOR = '#2166AC'
REFERENCE_LINE_COLOR = '#4D4D4D'

ablation = ablation_raw.copy()
kappa = kappa_raw.loc[
    kappa_raw['class_name'].eq('Basal-like')
].sort_values('kappa')

ablation = ablation.set_index('arm').loc[ARM_ORDER]
unique_tests = (
    ablation_tests.loc[
        ablation_tests['statistic'].eq('unique_node_fraction')
    ].set_index('panel').loc[ARM_ORDER]
)

fig5, axes = plt.subplots(
    2, 3,
    figsize=FIGSIZE['controls_sensitivity'],
    constrained_layout=True,
)
x = np.arange(len(ARM_ORDER))

# Panel A: reproducibilidad de las ablaciones
ax = axes[0, 0]
ax.bar(x, ablation['mean_pairwise_edge_jaccard'], color=ARM_COLORS)
ax.set_xticks(x, ARM_LABELS, rotation=25, ha='right')
ax.set_ylim(0, 0.68)
ax.set_ylabel('Mean pairwise edge Jaccard')
ax.set_title(FIG5_PANEL_TITLES[0], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel B: tamaño del núcleo estricto
ax = axes[0, 1]
bars = ax.bar(x, ablation['strict_edges'], color=ARM_COLORS)
for bar, value in zip(bars, ablation['strict_edges']):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 6,
        f'{int(value)}',
        ha='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xticks(x, ARM_LABELS, rotation=25, ha='right')
ax.set_ylim(0, 325)
ax.set_ylabel('Directed edges')
ax.set_title(FIG5_PANEL_TITLES[1], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel C: recuperación direccional corregida
ax = axes[0, 2]
bars = ax.bar(x, unique_tests['fold_over_null'], color=ARM_COLORS)
for bar, row in zip(bars, unique_tests.itertuples()):
    annotation = (
        'ns' if row.fdr_global >= 0.05
        else f'FDR={row.fdr_global:.3f}'
    )
    ax.text(
        bar.get_x() + bar.get_width()/2,
        row.fold_over_null + 0.07,
        annotation,
        ha='center',
        fontsize=7.8,
    )
ax.axhline(1.0, color=REFERENCE_LINE_COLOR, lw=0.9, ls='--')
ax.set_xticks(x, ARM_LABELS, rotation=25, ha='right')
ax.set_ylim(0, 3.25)
ax.set_ylabel('Unique-node fold over matched null')
ax.set_title(FIG5_PANEL_TITLES[2], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel D: comparador q con el mismo presupuesto de aristas
statistics = [
    'at_least_one_endpoint',
    'both_endpoints',
    'unique_node_fraction',
]
stat_labels = ['>=1 endpoint', 'Both endpoints', 'Unique nodes']
full = (
    consensus_tests.loc[
        consensus_tests['panel'].eq('Basal-like')
        & consensus_tests['term'].eq('Basal-like')
    ].set_index('statistic').loc[statistics]
)
q_raw = (
    q_tests.loc[
        q_tests['panel'].eq('q_raw::Basal-like')
        & q_tests['term'].eq('Basal-like')
    ].set_index('statistic').loc[statistics]
)
q_module = (
    q_tests.loc[
        q_tests['panel'].eq('q_module::Basal-like')
        & q_tests['term'].eq('Basal-like')
    ].set_index('statistic').loc[statistics]
)
ax = axes[1, 0]
stat_x = np.arange(3)
width = 0.25
ax.bar(
    stat_x - width,
    full['fold_over_null'],
    width,
    color=Q_COLORS['InterGATE'],
    label='InterGATE',
)
ax.bar(
    stat_x,
    q_raw['fold_over_null'],
    width,
    color=Q_COLORS['q raw'],
    label='q raw',
)
ax.bar(
    stat_x + width,
    q_module['fold_over_null'],
    width,
    color=Q_COLORS['q + module heuristic'],
    label='q + module heuristic',
)
ax.axhline(1.0, color=REFERENCE_LINE_COLOR, lw=0.9, ls='--')
ax.set_xticks(stat_x, stat_labels, rotation=15, ha='right')
ax.set_ylabel('Fold over matched null')
ax.set_title(FIG5_PANEL_TITLES[3], loc='left')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)

# Panel E: número de aristas y reproducibilidad a través de kappa
ax = axes[1, 1]
kappa_x = np.arange(len(kappa))
bars = ax.bar(
    kappa_x,
    kappa['strict_edges'],
    color='#C7C7C7',
    width=0.62,
    label='Strict edges',
)
ax.set_xticks(kappa_x, [f'{value:.3f}' for value in kappa['kappa']])
ax.set_xlabel('kappa')
ax.set_ylabel('Strict 5/5 edges', color=DARK_NEUTRAL)
ax_right = ax.twinx()
ax_right.plot(
    kappa_x,
    kappa['mean_pairwise_jaccard'],
    color=KAPPA_LINE_COLOR,
    marker='o',
    lw=2,
    label='Seed Jaccard',
)
ax_right.set_ylim(0.4, 0.8)
ax_right.set_ylabel('Mean pairwise Jaccard', color=KAPPA_LINE_COLOR)
ax.set_title(FIG5_PANEL_TITLES[4], loc='left')
ax.legend(
    [bars, ax_right.lines[0]],
    ['Strict edges', 'Seed Jaccard'],
    frameon=False,
    loc='upper left',
)
ax.spines['top'].set_visible(False)
ax_right.spines['top'].set_visible(False)

# Panel F: recuperación Basal-like en cada presupuesto
ax = axes[1, 2]
ax.plot(
    kappa_x,
    kappa['basal_at_least_one_endpoint_fold_over_null'],
    marker='o', lw=2, label='>=1 endpoint',
)
ax.plot(
    kappa_x,
    kappa['basal_both_endpoints_fold_over_null'],
    marker='s', lw=2, label='Both endpoints',
)
ax.plot(
    kappa_x,
    kappa['basal_unique_node_fraction_fold_over_null'],
    marker='^', lw=2, label='Unique nodes',
)
ax.axhline(1.0, color=REFERENCE_LINE_COLOR, lw=0.9, ls='--')
ax.set_xticks(kappa_x, [f'{value:.3f}' for value in kappa['kappa']])
ax.set_xlabel('kappa')
ax.set_ylabel('Fold over matched null')
ax.set_title(FIG5_PANEL_TITLES[5], loc='left')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)

fig5.suptitle(
    FIG5_TITLE,
    fontsize=STYLE['figure_title_size'],
    fontweight='bold',
)
export_figure(fig5, 'Figure_controls_sensitivity')
display(fig5)

## Comprobación final

Esta celda verifica que las cinco figuras existen en PDF, SVG y PNG. Los
objetos `fig1` a `fig5` siguen disponibles para continuar editándolos.

In [ ]:
figure_specs = [
    ('Figure_structural_recovery', 'fig1'),
    ('Figure_Basal_strict_core', 'fig2'),
    ('Figure_biological_validation', 'fig3'),
    ('Figure_external_structural_concordance', 'fig4'),
    ('Figure_controls_sensitivity', 'fig5'),
]

rows = []
for stem, object_name in figure_specs:
    formats = {
        extension: (FIGURES / f'{stem}.{extension}').is_file()
        for extension in ('pdf', 'svg', 'png')
    }
    rows.append({
        'figure': stem,
        'matplotlib_object': object_name,
        **formats,
        'status': 'complete' if all(formats.values()) else 'missing',
    })

status = pd.DataFrame(rows)
display(status)
if not status[['pdf', 'svg', 'png']].to_numpy().all():
    raise RuntimeError('Falta al menos un formato de alguna figura.')
print('Las cinco figuras están completas y siguen abiertas para edición.')